# wildfire API tutorial

This notebook demonstrates the **saved-model wildfire API and the heat-flux API** with the included FishCreek data. Run the cells from top to bottom using Python 3.12. Each execution creates a fresh output directory. The active default is the supported 5×5 checkpoint from the completed `20261010_categorical_balanced_benchmark` run. Earlier previews are historical and identify their own models.

Ordinary one-hour prediction returns one **`Prediction` object** with analytic probability, validity, rate, optional Earth coordinates, forecast horizon and cutoff timestamps. Recursive multi-hour prediction returns **`CumulativePrediction`**, with first-new-burn probability, total burned probability, Monte Carlo standard errors and pointwise nominal 95% Wilson bounds at explicitly requested positive whole-hour horizons. It advances binary fire states one hour at a time; newly ignited cells become sources during the following hour. The ordinary one-hour calculation and the heat walkthrough below retain their analytic probability. Wildfire plots display **`q = result.probability`**, the probability of new burning during the forecast interval, on a 0–1 scale. The hourly rate `lambda = result.rate` is a separate diagnostic. For the default exponential model at known unburned cells, `q = 1 - exp(-lambda * dt_hours)`; known already-burned cells have `q = 0`. Call `result.plot()` to show a matrix or `result.compare(...)` to combine next-hour event probabilities, reconstructed new burns, and the area already burned at the cutoff in one plot. Heat prediction continues to return **`(heat_matrix, coordinate_matrix)`** and provides detailed physical results separately.

Every geographic coordinate pair is a cell center, ordered **`(latitude, longitude)`**, in **WGS84 degrees (EPSG:4326)**. Coordinates remain static across time.

| Input organization | Wildfire probability field | Geographic coordinates |
| --- | --- | --- |
| Raw spatial snapshot | `(1, H, W)` | `(H, W, 2)`, or `None` without georeferencing |
| One-step dataset sequence | `(1, 1, H, W)` | `(H, W, 2)` |
| One sequence | `(T, 1, H, W)` | `(H, W, 2)` |
| Collated batch | `(B, T, 1, H, W)` | `(B, H, W, 2)` |
| One heat snapshot | `(H, W)` heat matrix | `(H, W, 2)` |

`B` is batch size, `T` time steps, `H` rows and `W` columns. Wildfire probabilities stay on the model device; coordinates are CPU `torch.float64` tensors. Heat matrices and coordinates are NumPy arrays.

The tutorial verifies data loading, prediction fields, correctly aligned matrix comparison, missingness, exports and save/reload. The data are retrospective burned-area reconstructions. This does not establish operational forecast skill or measured heat-flux accuracy. The physical heat parameters below are illustrative assumptions.

[Maintained API guide](README.md#python-interfaces) · [Complete option reference](documents/api.md) · [Heat guide](README.md#heat-guide)


## 1. Select the Python environment and workspace

From the directory containing `README.md` and `pyproject.toml`, the following shell commands install the notebook dependencies and open this file:

```sh
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[notebook]'
python -m jupyter lab api_tutorial.ipynb
```

Reuse an existing compatible environment when available. Select its Python 3.12 kernel in Jupyter. The notebook checks the actual interpreter below. It uses the workspace source directly, so edits to the API are reflected after a kernel restart.

The workspace is found by searching the kernel's working directory and its parents. If you launch Jupyter elsewhere, set the `WILDFIREGUARDIAN_ROOT` environment variable to the workspace path before opening it.

In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
import tempfile
from datetime import datetime, timedelta, timezone
from importlib.metadata import version

candidates = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("WILDFIREGUARDIAN_ROOT"):
    candidates.insert(0, Path(os.environ["WILDFIREGUARDIAN_ROOT"]).expanduser())
ROOT = next((p.resolve() for p in candidates
             if (p / 'pyproject.toml').is_file()
             and (p / 'wildfire').is_dir()), None)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from wildfire import (load_data, select_time, available_events, available_cutoffs,
                      load_model, train_model, prepare_training_data, collate_fire_sequences,
                      evaluate_cumulative, plot_matrix)
from wildfire.schema import TRAINING_SOURCE_CHANNELS
from torch.utils.data import DataLoader

torch.set_num_threads(4)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160, "font.size": 10})
RUNS_DIR = ROOT / "demonstration/api_tutorial/runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="run-", dir=RUNS_DIR))
print("Python:", sys.executable)
print("Workspace:", ROOT)
print("Outputs for this execution:", OUTPUT_DIR)
display(pd.DataFrame({"package": ["numpy", "torch", "pandas", "pyproj", "rasterio"],
                      "version": [version(p) for p in ["numpy", "torch", "pandas", "pyproj", "rasterio"]]}))

## 2. Choose data independently of a model

Event discovery and raster loading require no checkpoint. `TRAINING_SOURCE_CHANNELS` is a data schema with temperature, wind, slope, tree cover and biome sources. Categorical sources expand into named indicator channels. Selecting these six sources keeps the tutorial focused on the maintained 19 input fields; all loader options remain available.

`EVENT` identifies a fire. `CUTOFF` identifies the hour to inspect after loading. The full timeline option below does not filter loading to that cutoff.


In [ ]:
DATA_ROOT = ROOT / 'data/environment'
DATA_CHANNELS = TRAINING_SOURCE_CHANNELS  # A data choice, independent of any model.
EVENT = "WY-BTF-002416_FishCreek"
CUTOFF = "2024-08-23T00:00:00Z"
DEVICE = "cpu"


### 2.1 Discover available fire events and cutoffs

`available_events(...)` lists catalog event identifiers. `available_cutoffs(EVENT, ...)` lists hourly UTC cutoffs. Neither function loads a model or raster pixels. `sequence_length=2` restricts discovery to cutoffs with a following-hour endpoint, which is useful for comparing a prediction with reconstructed newly burned cells.


In [ ]:
EVENTS = available_events(data_root=DATA_ROOT, sequence_length=2)
assert EVENT in EVENTS, "The tutorial event needs a complete two-hour catalog window."
CUTOFFS = available_cutoffs(EVENT, data_root=DATA_ROOT, sequence_length=2)
assert CUTOFF in CUTOFFS, "The tutorial cutoff needs its following-hour endpoint."
EVENTS_AT_CUTOFF = available_events(
    CUTOFF, data_root=DATA_ROOT, sequence_length=2,
)
assert EVENT in EVENTS_AT_CUTOFF
print("Events with complete two-hour windows:", len(EVENTS))
display(pd.DataFrame({"available_event": EVENTS}))
print("Eligible cutoffs for", EVENT, ":", len(CUTOFFS))
display(pd.DataFrame({"available_cutoff_utc": CUTOFFS}).head(10))
print("Selected tutorial event and cutoff:", EVENT, CUTOFF)
print("Events available at this cutoff:", EVENTS_AT_CUTOFF)


## 3. Load a complete event timeline, then select a cutoff

The loader follows the PyTorch `Dataset` interface: construct a dataset, inspect its length or sample metadata, and index it to read tensors. `load_data(...)` is independent of the model. Its default is a 24-hour window; **`sequence_length=None` explicitly requests the full selected event timeline**. With one event and explicit **`patch_size=None`**, this produces one sample containing all available timestamps and the full original event rectangle. Environmental pixels outside unavailable arrival labels remain present. Arrival availability does not define the physical `pixel_valid` support. A `time_range=(start, end)` would restrict that timeline inclusively. Positive integer lengths and `temporal_stride` retain windowed loading.

Dataset construction reads metadata. Indexing materializes the selected tensors; this FishCreek example contains 1,639 hours on a 103 × 73 grid and its 19 feature channels occupy about 893 MiB before masks and labels. The context manager closes raster readers after loading. `select_time(event_data, CUTOFF)` selects a snapshot from the loaded timeline without opening files, preserves the time dimension, and shares static fields and tensor storage.

The sample contains retrospective arrival and outcome fields. Probability inference reads current state, environmental inputs, masks and geometry. Future labels are used only for comparison and evaluation. Ordinary one-hour prediction and heat retain unknown burned states. Recursive prediction below explicitly assumes unavailable initial burned states are unburned inside real pixel/time support; missing environmental inputs and padding remain unavailable. The original reference validity masks are preserved, so an assumed initial zero never becomes an observed negative outcome.


In [ ]:
with load_data(DATA_ROOT, events=[EVENT], channels=DATA_CHANNELS,
               sequence_length=None, patch_size=None) as dataset:
    assert len(dataset) == 1
    timeline_location = dataset.sample_location(0)
    event_data = dataset[0]  # All available event times, not only CUTOFF.

assert bool(event_data["time_valid"].all())
assert len(event_data["time_utc"]) == len(available_cutoffs(EVENT, data_root=DATA_ROOT))
sample = select_time(event_data, CUTOFF)
H, W = sample["burned_area"].shape[-2:]
display(pd.DataFrame([timeline_location]))
display(pd.DataFrame([
    {"object": "full timeline features", "shape": tuple(event_data["features"].shape)},
    {"object": "selected burned_area", "shape": tuple(sample["burned_area"].shape)},
    {"object": "selected features", "shape": tuple(sample["features"].shape)},
    {"object": "pixel_valid", "shape": tuple(sample["pixel_valid"].shape)},
    {"object": "selected time_valid", "shape": tuple(sample["time_valid"].shape)},
    {"object": "static projected coordinates", "shape": tuple(sample["coordinates"].shape)},
]))
print("Event:", sample["event_id"], "| selected cutoff:", CUTOFF, "| source EPSG:", int(sample["epsg"]))
display(pd.DataFrame({"channel_index": range(len(event_data["channel_names"])),
                      "channel_name": event_data["channel_names"]}))

# The event rectangle and environmental coverage are independent of arrival labels.
original_grid = json.loads((DATA_ROOT / "regions" / EVENT / "grid.json").read_text())
assert (H, W) == (int(original_grid["height"]), int(original_grid["width"]))
assert tuple(sample["features"].shape[-2:]) == (H, W)
assert bool(sample["pixel_valid"].all()) and bool(sample["time_valid"].all())
selected_feature_known = sample["feature_valid"][0] & torch.isfinite(sample["features"][0])
selected_arrival_known = sample["arrival_valid"][0] & torch.isfinite(sample["arrival_hours"][0])
outside_arrival = sample["pixel_valid"].reshape(H, W) & ~selected_arrival_known.reshape(H, W)
all_feature_channels_known = selected_feature_known.all(dim=0)
grid_summary = {
    "original_shape": [H, W], "rectangle_cells": H * W, "patch_size": None,
    "feature_channels": len(sample["channel_names"]),
    "all_features_known_cells": int(all_feature_channels_known.sum()),
    "outside_arrival_cells": int(outside_arrival.sum()),
    "all_features_known_outside_arrival_cells": int((all_feature_channels_known & outside_arrival).sum()),
}
display(pd.DataFrame([grid_summary]))
display(pd.DataFrame([
    {"channel": name, "valid_rectangle_cells": int(known.sum()),
     "valid_outside_arrival_cells": int((known & outside_arrival).sum())}
    for name, known in zip(sample["channel_names"], selected_feature_known)
]))
if EVENT == "WY-BTF-002416_FishCreek" and CUTOFF == "2024-08-23T00:00:00Z" and len(sample["channel_names"]) == 19:
    assert bool(all_feature_channels_known.all())  # This event/cutoff/schema, not every event.


## 4. Load a saved model independently and predict

`load_model()` restores the saved parameters, normalization and channel contract. Its default is the supported nonnegative 5×5 checkpoint from `20261010_categorical_balanced_benchmark`. `model.predict(sample)` selects its required named fields at the prediction boundary, including reordering a compatible sample or ignoring additional fields. Missing required fields are an error. The loaded data do not change.

The one-hour endpoint is first new cumulative burning during `(t, t + 1 hour]`. The ordinary `predict(..., dt_hours=h)` call and explicit fixed mode in `predict_cumulative(..., method="fixed")` rescale a frozen rate. The default cumulative method instead simulates recursive hourly spread. Section 4.4 demonstrates that default, and a separate fixed-mode example preserves fractional horizons and analytic rate diagnostics. `device="cpu"` is portable; `"auto"` selects an available supported accelerator.

Prediction returns probability, validity and rate tensors and static CPU WGS84 coordinates. `result.numpy()` supplies NumPy fields. Unknown real cells retain their locations; spatial padding receives NaN coordinates.


In [ ]:
from wildfire.paths import DEFAULT_CHECKPOINT
CHECKPOINT = DEFAULT_CHECKPOINT
model = load_model(CHECKPOINT, device=DEVICE)
display(pd.DataFrame([{
    "family": model.family, "device": str(model.device),
    "kernel_size": model.contract["kernel_size"], "channels": len(model.channel_names),
    "checkpoint_sha256": model.checkpoint_sha256,
}]))

result = model.predict(sample)
probability_tensor, coordinate_tensor = result.probability, result.coordinates
arrays = result.numpy()
probability, coordinates = arrays["probability"], arrays["coordinates"]
display(pd.DataFrame([
    {"output": "probability", "shape": probability.shape, "meaning": "next-hour new-burn probability"},
    {"output": "coordinates", "shape": coordinates.shape, "meaning": "latitude, longitude; WGS84 degrees"},
]))
row, column = H // 2, W // 2
latitude, longitude = coordinates[row, column]
print(f"Cell row={row}, column={column}: latitude={latitude:.8f}, longitude={longitude:.8f}")
print("Probability at this cell:", probability[0, 0, row, column])

### 4.1 Inspect probability and rate validity

`result` already contains `probability`, `valid`, `rate`, and `rate_valid`. Rates have units h⁻¹; use `probability` for probability and heat calculations.

This example has many unavailable predictions because required local context is incomplete. A known-unburned target with a complete source neighborhood in which every source is known unburned receives a valid structural zero when the required input context is available. The existing one-hour training/report protocol uses a burned-neighbor-contact cohort; the cumulative evaluator in section 4.7 has no such contact filter. No known burned neighbor is insufficient to establish zero when one or more source states are unknown. Coverage counts do not measure forecast accuracy; retain unavailable values as NaN. The notebook clears saved outputs after API changes; rerun it to generate outputs for the current implementation.

In [ ]:
details = result  # Every diagnostic is available from the original prediction.
valid = details.valid.cpu().numpy()
diagnostic_name = "rate"  # lambda diagnostics; plotting uses result.probability (q).
diagnostic = getattr(details, diagnostic_name).cpu().numpy()
torch.testing.assert_close(probability_tensor, details.probability, rtol=0, atol=0, equal_nan=True)
display(pd.DataFrame([{
    "known_predictions": int(valid.sum()), "unavailable_predictions": int((~valid).sum()),
    "total_cells": H * W, "minimum_known_probability": float(probability[valid].min()),
    "maximum_known_probability": float(probability[valid].max()),
}]))

real_unknown = (~valid[0, 0]) & sample["pixel_valid"][0].numpy()
if real_unknown.any():
    unknown_row, unknown_column = np.argwhere(real_unknown)[0]
    assert np.isfinite(coordinates[unknown_row, unknown_column]).all()
    print("Unavailable prediction at a real cell:", (int(unknown_row), int(unknown_column)))
    print("Its retained latitude/longitude:", coordinates[unknown_row, unknown_column])

### 4.2 Use your own arrays

`model.predict(burned_area, environmental_data, channel_names=..., ...)` accepts raw arrays and boolean validity masks. Preserve the saved channel order and physical units. Supply both `affine_transform` and source `epsg` for coordinates, or omit both for an array-only result with `coordinates=None`. The six affine coefficients `(a, b, c, d, e, f)` describe pixel corners; the API converts centers using `(column + 0.5, row + 0.5)`.

The assertions below verify raw-array and sample predictions agree, and that omitting geographic metadata changes only coordinate availability. Existing consumers that need a tuple can explicitly call `model.predict_coordinates(...)`.


In [ ]:
raw_result = model.predict(
    sample["burned_area"], sample["features"],
    channel_names=sample["channel_names"],
    burned_area_valid=sample["burned_area_valid"], feature_valid=sample["feature_valid"],
    affine_transform=sample["affine_transform"], epsg=sample["epsg"],
    pixel_valid=sample["pixel_valid"], time_valid=sample["time_valid"],
)
array_only_result = model.predict(
    sample["burned_area"], sample["features"], channel_names=model.channel_names,
    burned_area_valid=sample["burned_area_valid"], feature_valid=sample["feature_valid"],
)
for other in (raw_result, array_only_result):
    torch.testing.assert_close(other.probability, result.probability, rtol=0, atol=0, equal_nan=True)
    torch.testing.assert_close(other.valid, result.valid, rtol=0, atol=0)
torch.testing.assert_close(raw_result.coordinates, result.coordinates, rtol=0, atol=0, equal_nan=True)
print("Raw-array and sample predictions agree; georeferencing is optional for arrays.")


### 4.3 Plot a specific index from the loaded event timeline

Change `PLOT_TIME_INDEX` below to choose the input hour. It indexes **`event_data`**, the loaded timeline, with zero-based positions; negative values count from the end. `select_time(event_data, index=PLOT_TIME_INDEX)` selects the time, features, labels and masks together without reading files. It retains `T=1`.

After predicting that selection, `plot_result.plot()` plots its sole probability matrix `q = plot_result.probability`. Its local prediction index is zero, regardless of the original loaded time index. `plot_result.compare(event_data)` produces one combined plot: colors for next-hour new-burn probability `q` form the background, a translucent brown layer shows area already burned at the selected cutoff, and hollow cyan markers show reconstructed new burns in the following hour. Their interiors reveal the underlying `q` color, or gray where `q` is unavailable. The full loaded FishCreek grid remains visible: all 103 rows and 73 columns. A boundary labeled "Loaded region" outlines the sample's physical `pixel_valid` region, independently of prediction validity; the plot does not zoom to burned cells. It aligns `burned_area` at the exact cutoff and `newly_burned_area` at the exact next-hour endpoint in the full timeline. Missing reference timestamps remain unavailable for their own layers. Known new burns remain visible even where probability is unavailable; marker absence does not imply a negative outcome when the reference is unknown.

By contrast, when a prediction already contains multiple time steps, `sequence_result.plot(index=t)` selects its time step `t`; a batched result uses `batch_result.plot(index=(b, t))`. Those indices always address the prediction's own shape.


In [ ]:
PLOT_TIME_INDEX = 70  # Change this value to choose another loaded event hour.
plot_sample = select_time(event_data, index=PLOT_TIME_INDEX)
plot_result = model.predict(plot_sample)
plot_cutoff = pd.to_datetime(int(plot_sample["time_utc"][0]), unit="s", utc=True)
assert plot_result.probability.shape == (1, 1, H, W)
print("Loaded time index:", PLOT_TIME_INDEX, "| selected cutoff:", plot_cutoff.isoformat())
print("Prediction shape:", tuple(plot_result.probability.shape), "| local prediction time index: 0")

fig, ax = plot_result.plot(
    cmap="magma", title=f"Next-hour new-burn probability q\nCutoff: {plot_cutoff.isoformat()}",
)
fig.savefig(OUTPUT_DIR / "selected_time_probability.png", bbox_inches="tight")
plt.show()
plt.close(fig)

fig, axes = plot_result.compare(
    event_data, mode="overlay", figsize=(7, 6),
    titles=f"Previous burned area, next-hour new burns and q\nCutoff: {plot_cutoff.isoformat()}",
)
assert len(axes) == 1
np.testing.assert_allclose(axes[0].get_xlim(), (-0.5, W - 0.5))
np.testing.assert_allclose(axes[0].get_ylim(), (H - 0.5, -0.5))
fig.savefig(OUTPUT_DIR / "selected_time_events_vs_probability.png", bbox_inches="tight")
plt.show()
plt.close(fig)


### 4.4 Recursively predict first new burns at caller-selected whole hours

Enter **`HORIZONS_HOURS`** below. Recursive mode accepts distinct, finite, positive integer-valued durations, including `3.0`, and preserves their supplied order. It rejects fractional hours before inference. `CUMULATIVE_COMPARISON_HOURS` must be one of those durations. `EVALUATION_HOURS` is another **explicit caller selection** from the computed durations; evaluation never silently selects every available horizon. Do not supply `dt_hours` together with `hours`.

The default sampling budget is **`N_SAMPLES=1024`** independent trajectories; 256 and 4096 are useful alternative budgets, and any positive integer is accepted. `TRAJECTORY_BATCH_SIZE=32` limits trajectories processed together. These two controls exclude booleans; `SEED=0` is a nonnegative integer. Local CPU random streams preserve global random state. With the same seed and block size, changing horizon order or extending the horizon retains earlier results, and increasing the sample count preserves earlier simulated trajectories.

The initial state is binary. During each simulated hour the saved one-hour predictor evaluates that trajectory's state and current interval's environment, and independent conditional ignition draws update all cells **simultaneously**. Burning is absorbing. A cell ignited during one hour first contributes to spread during the following hour. For Bayesian checkpoints, one component is drawn using saved posterior weights for each trajectory and retained for its entire path; every component keeps its fitted preprocessing. The joint process assumes independent cell ignitions conditional on the current state, environment and component. No retraining is needed.

`MISSING_BURNED_AREA="unburned"` explicitly initializes unavailable initial burned states to zero **only inside real pixel/time support**. Set it to `"unknown"` to preserve the original unknown state. Unavailable environmental inputs and physical/time padding remain unavailable under both initial-state policies. The result records this policy and the number of assumed cutoff-cell entries in metadata. The original sample/reference tensors are not changed. The full original rectangle remains loaded with `patch_size=None`. This tutorial selects **`BOUNDARY="zero"`**, which is the recursive API default: source-neighborhood cells outside the supplied rectangle are treated as unburned. This is an explicit outside-fire assumption. It prevents boundary-related coverage from eroding inward as the horizon increases, while the output shape remains the original rectangle; it does not add output pixels. Missing environmental inputs **inside** the rectangle and physical/time padding remain unavailable. An explicit `boundary="unknown"` instead leaves outside-grid source states unresolved and can produce a growing unavailable gray border even where all environmental pixels inside the rectangle are valid. These exterior options do not fill internal environmental gaps.

`probability` estimates the chance of a **first new burn after the cutoff**. `burned_probability` includes cells already burned at the cutoff. Known initial burns have exact new-burn probability zero and total burned probability one on declared coverage. The horizon axis is separate from the input's time/batch dimensions: the shape is `(K, *base_prediction_shape)` for `K` requested durations, while static coordinates gain no horizon axis. Leading input dimensions remain independent cutoffs, not successive simulated states.

`mc_standard_error = sqrt(q_hat * (1 - q_hat) / N_SAMPLES)` and `confidence_lower`/`confidence_upper` are **pointwise nominal 95% Wilson bounds for new-burn probability**. They measure simulation error under this process, not model, observation, weather or joint-map uncertainty. Known initial burns receive exact zero errors and bounds `[0,0]`. At 1024 samples, the maximum theoretical standard error is `1 / (2 * sqrt(1024)) = 0.015625`. Recursive aggregate `rate` is NaN and `rate_valid` is false because one final hourly rate does not describe the cumulative interval.

The next call receives only the selected cutoff's input fields. Arrival and future outcome labels are omitted from the inference mapping. With no environmental forecast, the current environmental conditions repeat hourly. The reference timeline is supplied later only to comparison and evaluation.


In [ ]:
# Caller-entered controls: edit these before running this cell.
HORIZONS_HOURS = [1, 3, 6, 12]
CUMULATIVE_COMPARISON_HOURS = 6
EVALUATION_HOURS = [1, 3, 6]  # Required evaluation durations; each must exist above.
N_SAMPLES = 1024  # Alternatives include 256 and 4096.
SEED = 0
TRAJECTORY_BATCH_SIZE = 32
MISSING_BURNED_AREA = "unburned"  # Or "unknown"; environmental gaps stay unknown.
ENVIRONMENTAL_FORECAST = None  # Replace with a caller-supplied mapping; see 4.5.
BOUNDARY = "zero"  # Recursive default: outside-rectangle source cells are unburned.

from numbers import Real
def check_whole_hours(values, name):
    if not isinstance(values, (list, tuple)) or not values:
        raise ValueError(f"{name} must be an explicitly entered nonempty list or tuple.")
    if any(isinstance(h, (bool, np.bool_)) or not isinstance(h, Real)
           or not np.isfinite(h) or h <= 0 or not float(h).is_integer() for h in values):
        raise ValueError(f"{name} must contain finite positive whole-hour durations.")
    if len(set(values)) != len(values):
        raise ValueError(f"{name} must contain distinct durations.")
check_whole_hours(HORIZONS_HOURS, "HORIZONS_HOURS")
check_whole_hours(EVALUATION_HOURS, "EVALUATION_HOURS")
assert CUMULATIVE_COMPARISON_HOURS in HORIZONS_HOURS
assert set(EVALUATION_HOURS).issubset(HORIZONS_HOURS)

inference_keys = ("burned_area", "features", "channel_names", "burned_area_valid",
                  "feature_valid", "pixel_valid", "time_valid", "affine_transform",
                  "epsg", "time_utc", "event_id")
cumulative_input = {key: plot_sample[key] for key in inference_keys}
assert "arrival_hours" not in cumulative_input and "newly_burned_area" not in cumulative_input
cumulative_result = model.predict_cumulative(
    cumulative_input, hours=HORIZONS_HOURS,  # method="recursive" is the default.
    n_samples=N_SAMPLES, seed=SEED, trajectory_batch_size=TRAJECTORY_BATCH_SIZE,
    missing_burned_area=MISSING_BURNED_AREA, environmental_forecast=ENVIRONMENTAL_FORECAST,
    boundary=BOUNDARY,
)
assert tuple(cumulative_result.hours) == tuple(HORIZONS_HOURS)
assert cumulative_result.probability.shape == (len(HORIZONS_HOURS), *plot_result.probability.shape)
assert torch.isnan(cumulative_result.rate).all() and not cumulative_result.rate_valid.any()
assert tuple(cumulative_result.coordinates.shape) == (H, W, 2)

cutoff_support = torch.broadcast_to(plot_sample["pixel_valid"].unsqueeze(-3), plot_sample["burned_area"].shape)
cutoff_support = cutoff_support & plot_sample["time_valid"].reshape(-1, 1, 1, 1)
original_known_state = plot_sample["burned_area_valid"] & torch.isfinite(plot_sample["burned_area"])
assumed_unburned = cutoff_support & ~original_known_state
known_initial_burn = cutoff_support & original_known_state & (plot_sample["burned_area"] == 1)
expected_assumed_count = int(assumed_unburned.sum()) if MISSING_BURNED_AREA == "unburned" else 0
assert cumulative_result.metadata["missing_burned_area"] == MISSING_BURNED_AREA
assert cumulative_result.metadata["assumed_unburned_cells"] == expected_assumed_count

cumulative_summary = []
for hours in cumulative_result.hours:
    horizon_result = cumulative_result.at(hours=hours)
    horizon_arrays = horizon_result.numpy()
    horizon_values, horizon_valid = horizon_arrays["probability"], horizon_arrays["valid"]
    known_values = horizon_values[horizon_valid]
    known_burn_mask = known_initial_burn.to(horizon_result.valid.device) & horizon_result.valid
    assert torch.all(horizon_result.probability[known_burn_mask] == 0)
    assert torch.all(horizon_result.burned_probability[known_burn_mask] == 1)
    assert torch.all(horizon_result.mc_standard_error[known_burn_mask] == 0)
    assert torch.all(horizon_result.confidence_lower[known_burn_mask] == 0)
    assert torch.all(horizon_result.confidence_upper[known_burn_mask] == 0)
    assert np.all((known_values >= 0) & (known_values <= 1))
    for name in ("burned_probability", "mc_standard_error", "confidence_lower", "confidence_upper"):
        assert horizon_arrays[name].shape == horizon_values.shape
        assert np.isnan(horizon_arrays[name][~horizon_valid]).all()
    cumulative_summary.append({
        "hours_after_cutoff": hours, "known_probability_cells": int(horizon_valid.sum()),
        "minimum_known_q": float(known_values.min()) if known_values.size else None,
        "maximum_known_q": float(known_values.max()) if known_values.size else None,
        "maximum_mc_standard_error": float(horizon_arrays["mc_standard_error"][horizon_valid].max()) if known_values.size else None,
    })
for earlier, later in zip(sorted(HORIZONS_HOURS)[:-1], sorted(HORIZONS_HOURS)[1:]):
    left, right = cumulative_result.at(hours=earlier), cumulative_result.at(hours=later)
    shared = left.valid & right.valid
    assert torch.all(left.probability[shared] <= right.probability[shared])
display(pd.DataFrame(cumulative_summary))
print("Recursive initialization and sampling metadata:", cumulative_result.metadata)

# Ordinary predict remains analytic. Compare the same initialized cutoff/environment.
analytic_input = dict(cumulative_input)
if MISSING_BURNED_AREA == "unburned":
    analytic_input["burned_area"] = torch.where(assumed_unburned, 0, plot_sample["burned_area"])
    analytic_input["burned_area_valid"] = plot_sample["burned_area_valid"] | assumed_unburned
if ENVIRONMENTAL_FORECAST is not None:
    analytic_input["features"] = ENVIRONMENTAL_FORECAST["features"][0]
    analytic_input["channel_names"] = ENVIRONMENTAL_FORECAST["channel_names"]
    if ENVIRONMENTAL_FORECAST.get("feature_valid") is None:
        analytic_input.pop("feature_valid", None)
    else:
        analytic_input["feature_valid"] = ENVIRONMENTAL_FORECAST["feature_valid"][0]
analytic_one_hour = model.predict(analytic_input, boundary=BOUNDARY)
if 1 in HORIZONS_HOURS:
    recursive_one_hour = cumulative_result.at(hours=1)
    common = analytic_one_hour.valid & recursive_one_hour.valid
    compared_cells = int(common.sum())
    max_difference = float(torch.max(torch.abs(analytic_one_hour.probability[common] - recursive_one_hour.probability[common]))) if compared_cells else None
    # Hoeffding + a union bound: nominal >=99% agreement over this compared grid.
    simultaneous_tolerance = float(np.sqrt(np.log(2 * max(compared_cells, 1) / 0.01) / (2 * N_SAMPLES)))
    if compared_cells:
        assert max_difference <= simultaneous_tolerance
    print("One-hour analytic versus MC:", {"compared_cells": compared_cells,
          "maximum_absolute_difference": max_difference, "99pct_Hoeffding_union_tolerance": simultaneous_tolerance})

fig, axes = plt.subplots(1, len(HORIZONS_HOURS), figsize=(5 * len(HORIZONS_HOURS), 6),
                         squeeze=False, constrained_layout=True)
for ax, hours in zip(axes[0], cumulative_result.hours):
    cumulative_result.plot(hours=hours, index=0, ax=ax, cmap="magma", colorbar=False,
                           title=f"First new burn by {hours:g} h")
fig.colorbar(axes[0, -1].images[0], ax=list(axes[0]), label="Estimated first-new-burn probability")
fig.suptitle(f"Recursive hourly spread; cutoff: {plot_cutoff.isoformat()}")
fig.savefig(OUTPUT_DIR / "cumulative_probability_horizons.png", bbox_inches="tight")
plt.show()
plt.close(fig)

selected_horizon = cumulative_result.at(hours=CUMULATIVE_COMPARISON_HOURS)
selected_arrays = selected_horizon.numpy()
fig, ax = plot_matrix(selected_arrays["burned_probability"], valid=selected_arrays["valid"], index=0,
                      cmap="magma", vmin=0, vmax=1,
                      title=f"Total burned probability at {CUMULATIVE_COMPARISON_HOURS:g} h\nIncludes cells already burned at cutoff",
                      colorbar_label="Estimated total burned probability")
fig.savefig(OUTPUT_DIR / "cumulative_total_burned_probability.png", bbox_inches="tight")
plt.show()
plt.close(fig)

# Exact cutoff/endpoint references appear only in comparison and evaluation.
fig, axes = cumulative_result.compare(
    event_data, hours=CUMULATIVE_COMPARISON_HOURS, index=0, figsize=(7, 6),
    titles=(f"Previous burned area, next-{CUMULATIVE_COMPARISON_HOURS:g}-hour new burns and q"
            f"\nCutoff: {plot_cutoff.isoformat()}"),
)
assert len(axes) == 1
np.testing.assert_allclose(axes[0].get_xlim(), (-0.5, W - 0.5))
np.testing.assert_allclose(axes[0].get_ylim(), (H - 0.5, -0.5))
fig.savefig(OUTPUT_DIR / "cumulative_events_vs_probability.png", bbox_inches="tight")
plt.show()
plt.close(fig)

cumulative_arrays = cumulative_result.numpy()  # Independent CPU NumPy copies.
second_export = cumulative_result.numpy()
for name in cumulative_arrays:
    assert not np.shares_memory(cumulative_arrays[name], second_export[name])
assert json.loads(str(cumulative_arrays["metadata_json"])) == cumulative_result.metadata
CUMULATIVE_PROBABILITY_FILE = OUTPUT_DIR / "cumulative_probability.npz"
with CUMULATIVE_PROBABILITY_FILE.open("xb") as stream:
    np.savez_compressed(stream, **cumulative_arrays)
with np.load(CUMULATIVE_PROBABILITY_FILE, allow_pickle=False) as saved:
    for name, values in cumulative_arrays.items():
        np.testing.assert_array_equal(saved[name], values)
    assert json.loads(str(saved["metadata_json"])) == cumulative_result.metadata


### 4.5 Optional caller-supplied hourly environmental forecasts

Set `ENVIRONMENTAL_FORECAST` **before** the main recursive call to a mapping with `features`, `channel_names`, and optional boolean `feature_valid`. After named-channel selection, the complete feature shape is **`(S, *L, C, H, W)`**, where `L` contains the initial input's leading dimensions. For this one-step sample, it is `(S, 1, C, H, W)`; the separate `S` axis indexes forecast intervals and is not the original `T` axis. Slice zero supplies `(t, t+1 hour]`, slice one the next interval, and so on. Require `S >= max(HORIZONS_HOURS)`; additional slices are ignored. A supplied forecast mask must match the complete feature shape, including all channels and slices. Saved named-channel ordering and categorical one-hot validation still apply.

The optional runnable example below constructs a **repeated-current scenario**, reorders named channels, and includes an extra interval to show the input organization. Replace these tensors with externally supplied hourly forecasts for a forecast scenario. It never reads future environmental observations from the retrospective reference timeline. Missing forecast inputs remain unavailable, and coverage evolves conservatively each hour, independently of whether sampled trajectories happen to ignite a cell.


In [ ]:
RUN_FORECAST_EXAMPLE = False  # Optional extra recursive call; replace with caller forecasts.
forecast_demo_metadata = None
if RUN_FORECAST_EXAMPLE:
    S = int(max(HORIZONS_HOURS)) + 1  # One extra slice is harmless.
    channel_order = torch.arange(len(cumulative_input["channel_names"]) - 1, -1, -1)
    hourly_features = cumulative_input["features"].unsqueeze(0).repeat(S, 1, 1, 1, 1)
    hourly_feature_valid = cumulative_input["feature_valid"].unsqueeze(0).repeat(S, 1, 1, 1, 1)
    caller_forecast = {
        "features": hourly_features.index_select(-3, channel_order),
        "channel_names": [cumulative_input["channel_names"][int(i)] for i in channel_order],
        "feature_valid": hourly_feature_valid.index_select(-3, channel_order),
    }
    assert caller_forecast["features"].shape == (S, 1, len(cumulative_input["channel_names"]), H, W)
    assert caller_forecast["feature_valid"].shape == caller_forecast["features"].shape
    forecast_result = model.predict_cumulative(
        cumulative_input, hours=HORIZONS_HOURS, environmental_forecast=caller_forecast,
        n_samples=N_SAMPLES, seed=SEED, trajectory_batch_size=TRAJECTORY_BATCH_SIZE,
        missing_burned_area=MISSING_BURNED_AREA, boundary=BOUNDARY,
    )
    forecast_demo_metadata = forecast_result.metadata
    if ENVIRONMENTAL_FORECAST is None:
        torch.testing.assert_close(forecast_result.probability, cumulative_result.probability,
                                   rtol=0, atol=0, equal_nan=True)
        torch.testing.assert_close(forecast_result.valid, cumulative_result.valid, rtol=0, atol=0)
    print("Caller forecast shape and environment policy:", tuple(caller_forecast["features"].shape),
          forecast_result.metadata["environment_policy"])
else:
    print("Optional caller-forecast example skipped; main-call environment policy:",
          cumulative_result.metadata["environment_policy"])


### 4.6 Explicit fixed-mode compatibility, including fractional durations

`method="fixed"` retains the earlier calculation with the current binary burned state, environment and model component rates held fixed. It accepts distinct positive real durations such as `0.5` or `3.5` hours. For this exponential checkpoint, a known unburned cell has `q(h)=1-exp(-lambda*h)` and a known burned cell has `q(h)=0`. Bayesian fixed mode averages each component's horizon probability and retains its existing equivalent interval rate diagnostic.

Fixed mode retains `rate`/`rate_valid`, adds `burned_probability`, and has no Monte Carlo fields. Sampling controls have no effect. It rejects environmental forecasts, and its original unknown burned-state behavior is preserved even though recursive mode's default initialization is unburned. Ordinary `model.predict(..., dt_hours=h)` remains the corresponding analytic single-horizon call. This example is separate from the recursive result used for evaluation.

This fixed compatibility example deliberately passes the chosen `boundary=BOUNDARY`, so it also uses outside-unburned source padding. Fixed-mode callers that omit `boundary` retain the original unknown-exterior default. Its original missing-internal-burn-state behavior is unchanged.


In [ ]:
FIXED_HORIZONS_HOURS = [0.5, 1, 3.5]
fixed_result = model.predict_cumulative(cumulative_input, hours=FIXED_HORIZONS_HOURS,
                                        method="fixed", boundary=BOUNDARY)
assert fixed_result.mc_standard_error is None
assert fixed_result.confidence_lower is None and fixed_result.confidence_upper is None
for hours in FIXED_HORIZONS_HOURS:
    fixed_at_hour = fixed_result.at(hours=hours)
    analytic_at_hour = model.predict(cumulative_input, dt_hours=hours, boundary=BOUNDARY)
    for field in ("probability", "valid", "rate", "rate_valid"):
        torch.testing.assert_close(getattr(fixed_at_hour, field), getattr(analytic_at_hour, field),
                                   rtol=0, atol=0, equal_nan=True)
    fixed_initial_burn = known_initial_burn.to(fixed_at_hour.valid.device) & fixed_at_hour.valid
    assert torch.all(fixed_at_hour.probability[fixed_initial_burn] == 0)
    assert torch.all(fixed_at_hour.burned_probability[fixed_initial_burn] == 1)
    if model.family == "exponential":
        original_unburned = (original_known_state & (plot_sample["burned_area"] == 0)).to(fixed_at_hour.valid.device)
        rate_known = fixed_at_hour.valid & fixed_at_hour.rate_valid & original_unburned
        torch.testing.assert_close(fixed_at_hour.probability[rate_known],
                                   -torch.expm1(-fixed_at_hour.rate[rate_known] * hours),
                                   rtol=1e-5, atol=1e-7)
fixed_arrays = fixed_result.numpy()
FIXED_CUMULATIVE_PROBABILITY_FILE = OUTPUT_DIR / "fixed_cumulative_probability.npz"
with FIXED_CUMULATIVE_PROBABILITY_FILE.open("xb") as stream:
    np.savez_compressed(stream, **fixed_arrays)
with np.load(FIXED_CUMULATIVE_PROBABILITY_FILE, allow_pickle=False) as saved:
    for name, values in fixed_arrays.items():
        np.testing.assert_array_equal(saved[name], values)
print("Fixed fractional horizons and analytic rate diagnostics verified:", fixed_result.hours)


### 4.7 Evaluate explicitly selected cumulative durations

The two public routes are `cumulative_result.evaluate(reference, hours=EVALUATION_HOURS, ...)` and `evaluate_cumulative(cumulative_result, reference, hours=EVALUATION_HOURS, ...)`. **`hours` is a required keyword**; every entered duration must already exist in the prediction, and its supplied order is retained. A scalar, such as `hours=6`, requests one duration. Select the endpoint exactly at cutoff plus that duration. A missing endpoint produces unavailable references and, for an empty cohort, `None` scores rather than invented targets.

The reference window below is selected from the separately held full timeline only **after prediction** and retains its original masks. `quantity="new_burn"` compares `probability` with first new burning on cells known unburned at the cutoff and known at the endpoint; it excludes initial burns. `quantity="burned"` compares `burned_probability` with the known endpoint cumulative state, including initial burns, without requiring a known cutoff reference. Total-burned scores therefore include deterministic cells already burned and measure a different target/cohort from future spread. Neither quantity requires a known burned neighbor at the initial cutoff. Unavailable reference labels stay excluded even where simulation assumed an initial unburned cell. Known reference transitions from burned to unburned are rejected.

`coverage="per_horizon"` uses each selected duration's covered reference cohort. `"shared"` intersects covered cohorts across **the explicitly selected durations**, so comparing durations uses the same entries. Each report still records its own `reference_eligible`, `prediction_covered` and final `evaluated` cell counts and weights, weighted coverage fractions and exclusions. `weight=None` means equal cell-cutoff weights; an optional finite nonnegative scalar, spatial `(H,W)` map, or exact base-probability-shape array has no horizon axis. Zero weights exclude entries. An optional boolean `cohort_mask` has spatial or exact base shape. Cells/cutoffs are aggregated within each duration; durations are never pooled.

The reports include Brier and class-specific Brier scores, exact negative log-likelihood, separately clipped log loss at `1e-7`, average precision, ROC AUC, trapezoidal precision-recall AUC, exact/floored Bernoulli KL, and seven Bernoulli distance summaries. These distances compare each binary outcome distribution with its predicted Bernoulli distribution on `{0,1}`: total variation, Hellinger, Jensen-Shannon divergence and distance, Bhattacharyya, and Wasserstein-1/-2 with unit distance between zero and one. They are not geographic displacement or a joint distribution distance between entire fire maps. With a binary reference, exact KL equals exact negative log-likelihood. An unsupported zero-probability positive or one-probability negative has infinite exact loss, exported as the JSON string `"infinity"`; separately clipped scores do not conceal this.

The separate `classification` row uses the caller's fixed `EVALUATION_THRESHOLD`, returning weighted TP/FP/TN/FN, precision, recall, F1, MCC, accuracy, specificity, balanced accuracy and intersection-over-union. The metrics block also retains the original diagnostic scores at threshold 0.5. No threshold is tuned to these outcomes. `calibration` reports fixed reliability bins and weighted expected calibration error. Empty cohorts have zero confusion counts, `None` scores and an empty bin list. Monte Carlo Wilson bounds are not confidence intervals for these evaluation metrics. This single-event retrospective example demonstrates the evaluation API, not external forecast skill.


In [ ]:
EVALUATION_THRESHOLD = 0.5  # Caller chosen; never fitted to reference outcomes.
EVALUATION_WEIGHT = None
EVALUATION_COHORT_MASK = None
evaluation_reference_end = plot_cutoff + pd.Timedelta(hours=max(EVALUATION_HOURS))
evaluation_reference = select_time(event_data, plot_cutoff.isoformat(),
                                   evaluation_reference_end.isoformat())
assert evaluation_reference["burned_area"].shape[-2:] == cumulative_result.probability.shape[-2:]

cumulative_evaluations = {}
evaluation_tables, calibration_rows = [], []
for quantity in ("new_burn", "burned"):
    for coverage_policy in ("per_horizon", "shared"):
        report = cumulative_result.evaluate(
            evaluation_reference, hours=EVALUATION_HOURS, quantity=quantity,
            coverage=coverage_policy, threshold=EVALUATION_THRESHOLD,
            weight=EVALUATION_WEIGHT, cohort_mask=EVALUATION_COHORT_MASK,
        )
        assert [row["hours"] for row in report["per_horizon"]] == EVALUATION_HOURS
        assert report["protocol"]["cutoff_neighbor_contact_required"] is False
        assert all(row["metrics"]["sample_count"] == row["coverage"]["evaluated_cells"]
                   for row in report["per_horizon"])
        if coverage_policy == "shared":
            assert len({row["coverage"]["evaluated_cells"] for row in report["per_horizon"]}) == 1
        name = f"{quantity}_{coverage_policy}"
        cumulative_evaluations[name] = report
        table = pd.json_normalize(report["per_horizon"])
        table.insert(0, "coverage_policy", coverage_policy)
        table.insert(0, "quantity", quantity)
        evaluation_tables.append(table)
        for row in report["per_horizon"]:
            for bin_row in row["calibration"]["bins"]:
                calibration_rows.append({"quantity": quantity, "coverage_policy": coverage_policy,
                                         "hours": row["hours"], **bin_row})

# The standalone public function produces the same report as the result method.
standalone_report = evaluate_cumulative(
    cumulative_result, evaluation_reference, hours=EVALUATION_HOURS,
    quantity="new_burn", coverage="per_horizon", threshold=EVALUATION_THRESHOLD,
    weight=EVALUATION_WEIGHT, cohort_mask=EVALUATION_COHORT_MASK,
)
assert standalone_report == cumulative_evaluations["new_burn_per_horizon"]
evaluation_table = pd.concat(evaluation_tables, ignore_index=True)

identity_columns = ["quantity", "coverage_policy", "hours"]
display(evaluation_table[identity_columns + [
    "coverage.reference_eligible_cells", "coverage.prediction_covered_cells", "coverage.evaluated_cells",
    "coverage.coverage_fraction", "coverage.prediction_coverage_fraction",
    "coverage.initially_burned_excluded_cells", "coverage.excluded_by_shared_coverage_cells",
]])
display(evaluation_table[identity_columns + [
    "metrics.brier", "metrics.exact_negative_log_likelihood_mean", "metrics.clipped_log_loss",
    "metrics.average_precision", "metrics.roc_auc", "metrics.precision_recall_auc",
    "classification.f1", "classification.mcc", "classification.intersection_over_union",
    "calibration.weighted_expected_calibration_error",
]])
distance_fields = ["bernoulli_kl_divergence_mean", "floored_bernoulli_kl_divergence_mean",
                   "bernoulli_total_variation_distance_mean", "bernoulli_hellinger_distance_mean",
                   "bernoulli_jensen_shannon_divergence_mean", "bernoulli_jensen_shannon_distance_mean",
                   "bernoulli_bhattacharyya_distance_mean", "bernoulli_wasserstein_1_distance_mean",
                   "bernoulli_wasserstein_2_distance_mean"]
display(evaluation_table[identity_columns + [f"metrics.{field}" for field in distance_fields]])

CUMULATIVE_EVALUATION_FILE = OUTPUT_DIR / "cumulative_evaluation.json"
evaluation_payload = {
    "event": EVENT, "cutoff_utc": plot_cutoff.isoformat(),
    "reference_end_utc": evaluation_reference_end.isoformat(),
    "explicit_evaluation_hours": EVALUATION_HOURS,
    "prediction_metadata": cumulative_result.metadata, "reports": cumulative_evaluations,
}
CUMULATIVE_EVALUATION_FILE.write_text(json.dumps(evaluation_payload, indent=2, allow_nan=False) + "\n")
evaluation_table.drop(columns=["calibration.bins"], errors="ignore").to_csv(
    OUTPUT_DIR / "cumulative_evaluation.csv", index=False)
calibration_table = pd.DataFrame(calibration_rows, columns=[
    "quantity", "coverage_policy", "hours", "lower", "upper", "upper_inclusive", "sample_count",
    "positive_sample_count", "represented_population_weight", "population_fraction",
    "mean_probability", "weighted_outcome_rate",
])
calibration_table.to_csv(OUTPUT_DIR / "cumulative_calibration_bins.csv", index=False)
def reject_nonstandard_json(value):
    raise ValueError(f"Unexpected JSON constant: {value}")
saved_evaluation = json.loads(CUMULATIVE_EVALUATION_FILE.read_text(), parse_constant=reject_nonstandard_json)
assert saved_evaluation == evaluation_payload
saved_evaluation_csv = pd.read_csv(OUTPUT_DIR / "cumulative_evaluation.csv")
saved_calibration_csv = pd.read_csv(OUTPUT_DIR / "cumulative_calibration_bins.csv")
assert len(saved_evaluation_csv) == 4 * len(EVALUATION_HOURS)
assert len(saved_calibration_csv) == len(calibration_rows)
print("Verified explicit-hour evaluation exports:", CUMULATIVE_EVALUATION_FILE)


### 4.8 Optional: a saved random forest through the same public API

Set `RUN_FOREST_EXAMPLE=True` to run this separate example. `RF_HOURS=6` is an explicitly entered, editable cumulative prediction **and evaluation** duration. `RF_N_SAMPLES=256` controls simulation effort. This example uses the same selected cutoff and full rectangle as the main demonstration, with its own saved forest, preprocessing and CPU inference. It does not change `model`, `cumulative_result`, or the heat example.

`load_model("...joblib", device="cpu")` restores a fitted `PatchRandomForest`. Every hourly transition uses the forest's **mean tree probability** given the current binary state and environmental inputs. Trees are not sampled as Bayesian components. Newly ignited cells become sources in the following hour; a complete neighborhood with no initial fire has structural probability zero. Missing initial burned states are assumed unburned, exterior source cells are padded unburned, and unavailable interior environmental inputs remain masked. Without an environmental forecast, the cutoff conditions repeat at each hour.

The two maps below are different outputs: `probability` estimates a first **new burn since the cutoff**, whereas `burned_probability` includes cells already burned at that cutoff. Original burned cells therefore have values zero and one, respectively. Both retain the original rectangle and coordinates. Recursive rates remain NaN with `rate_valid=False`. Ordinary forest `predict()` has a fitted one-hour endpoint; its equivalent diagnostic `-log(1-q)` is not a fitted continuous-time hazard. Forest `method="fixed"` accepts only `hours=1`; use recursion for longer intervals.

References are selected only after forecasting and retain their original validity masks. The two evaluation tables include all **nine** exact/floored Bernoulli KL and probability-distance fields: exact KL, floored KL, total variation, Hellinger, Jensen-Shannon divergence, Jensen-Shannon distance, Bhattacharyya distance, Wasserstein-1 and Wasserstein-2. These are per-cell binary-distribution comparisons, not geographic distances. This optional single-event example demonstrates the API; it does not establish held-out forecast performance.


In [ ]:
RUN_FOREST_EXAMPLE = False  # Optional: no extra forest simulation in the default walkthrough.
RF_HOURS = 6                # Editable cumulative prediction AND explicit evaluation duration.
RF_N_SAMPLES = 256          # Increase for smaller Monte Carlo error.
FOREST_CHECKPOINT = ROOT / "runs/20261010_categorical_balanced_benchmark/forests/rf5_depth8.joblib"
forest_demo_metadata = None
forest_demo_controls = {
    "checkpoint": str(FOREST_CHECKPOINT.relative_to(ROOT)), "hours": RF_HOURS,
    "n_samples": RF_N_SAMPLES, "seed": SEED,
    "trajectory_batch_size": TRAJECTORY_BATCH_SIZE,
    "boundary": BOUNDARY, "missing_burned_area": MISSING_BURNED_AREA,
}
if RUN_FOREST_EXAMPLE:
    check_whole_hours([RF_HOURS], "RF_HOURS")
    forest_model = load_model(FOREST_CHECKPOINT, device="cpu")
    assert forest_model.family == "random_forest" and str(forest_model.device) == "cpu"
    forest_input = dict(cumulative_input)  # Cutoff inputs only; no future outcomes or arrival labels.
    forest_result = forest_model.predict_cumulative(
        forest_input, hours=RF_HOURS, method="recursive", n_samples=RF_N_SAMPLES,
        seed=SEED, trajectory_batch_size=TRAJECTORY_BATCH_SIZE,
        missing_burned_area=MISSING_BURNED_AREA, boundary=BOUNDARY,
    )
    forest_horizon = forest_result.at(hours=RF_HOURS)
    forest_arrays = forest_horizon.numpy()
    assert forest_horizon.probability.shape == plot_result.probability.shape
    assert torch.isnan(forest_result.rate).all() and not forest_result.rate_valid.any()
    forest_demo_metadata = forest_result.metadata
    display(pd.DataFrame([{
        "family": forest_model.family, "device": str(forest_model.device),
        "kernel_size": forest_model.contract["kernel_size"], "hours": RF_HOURS,
        "n_samples": RF_N_SAMPLES, "valid_cells": int(forest_horizon.valid.sum()),
        "preprocessing": forest_model.contract["preprocessing_policy"],
    }]))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    for ax, field, title in zip(
        axes, ("probability", "burned_probability"),
        (f"Forest: first new burn by {RF_HOURS:g} h",
         f"Forest: total burned at {RF_HOURS:g} h\nIncludes burns already present at cutoff"),
    ):
        plot_matrix(forest_arrays[field], valid=forest_arrays["valid"], index=0,
                    ax=ax, cmap="magma", vmin=0, vmax=1, title=title,
                    colorbar_label="Estimated probability")
    fig.suptitle(f"Saved forest; cutoff: {plot_cutoff.isoformat()}")
    fig.savefig(OUTPUT_DIR / "forest_cumulative_probability.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)

    forest_reference_end = plot_cutoff + pd.Timedelta(hours=RF_HOURS)
    forest_reference = select_time(event_data, plot_cutoff.isoformat(),
                                   forest_reference_end.isoformat())
    forest_reports, forest_tables = {}, []
    for quantity in ("new_burn", "burned"):
        forest_report = evaluate_cumulative(
            forest_result, forest_reference, hours=RF_HOURS, quantity=quantity,
            coverage="per_horizon", threshold=EVALUATION_THRESHOLD,
        )
        assert forest_report["protocol"]["hours"] == [RF_HOURS]
        forest_reports[quantity] = forest_report
        table = pd.json_normalize(forest_report["per_horizon"])
        table.insert(0, "quantity", quantity)
        forest_tables.append(table)
    forest_table = pd.concat(forest_tables, ignore_index=True)
    display(forest_table[["quantity", "hours", "coverage.evaluated_cells", "metrics.brier",
                          "classification.f1", "calibration.weighted_expected_calibration_error"]])
    display(forest_table[["quantity", "hours"] + [f"metrics.{field}" for field in distance_fields]])
    forest_table.drop(columns=["calibration.bins"], errors="ignore").to_csv(
        OUTPUT_DIR / "forest_cumulative_evaluation.csv", index=False)
    forest_payload = {
        "event": EVENT, "cutoff_utc": plot_cutoff.isoformat(),
        "checkpoint_sha256": forest_model.checkpoint_sha256,
        "explicit_evaluation_hours": [RF_HOURS], "prediction_metadata": forest_demo_metadata,
        "reports": forest_reports,
    }
    (OUTPUT_DIR / "forest_cumulative_evaluation.json").write_text(
        json.dumps(forest_payload, indent=2, allow_nan=False) + "\n")
    with (OUTPUT_DIR / "forest_cumulative_probability.npz").open("xb") as stream:
        np.savez_compressed(stream, **forest_result.numpy())
    print("Forest probability maps and evaluation exports:", OUTPUT_DIR)
else:
    print("Optional forest example skipped; set RUN_FOREST_EXAMPLE=True to execute it.")


### 4.9 Evaluate every saved tensor model and random forest, and generate comparison plots

Run this command **in a terminal from the workspace root**, separately from the notebook. Edit `EVAL_HOURS` to change both the cumulative simulation duration and the exact reference endpoint. The specified real-data experiment roots currently contain **561 tensor checkpoints and 12 saved forests**. `--selection all` includes every canonical model. The runner restores each checkpoint's own preprocessing, uses `boundary="zero"` with the full input rectangle, and evaluates events held out by every selected model. For a forest whose saved contract lacks event splits, it records the split provenance recovered from a companion tensor contract with the same saved cache metadata hash and ordered channels.

```bash
EVAL_HOURS=6

.venv/bin/python -m wildfire evaluate-cumulative \
  --hours "$EVAL_HOURS" \
  --models-root runs/20261010_categorical_balanced_benchmark \
  --models-root runs/20261009_multiscale_mps_lr_0p001_0p01_0p1_floor_0p1 \
  --models-root runs/20261011_bayesian_kernel_average \
  --selection all \
  --split test \
  --n-samples 1024 \
  --seed 0 \
  --trajectory-batch-size 32 \
  --threads 4 \
  --device cpu \
  --threshold 0.5 \
  --output-dir "reports/cumulative_${EVAL_HOURS}h_all_models_with_forests"
```

The output directory must be new or empty. The exhaustive run evaluates every complete selected cutoff–endpoint interval and can require substantial computation. To inspect the model/case plan without simulation, add `--plan-only` and choose a separate output directory. For a bounded diagnostic, add `--event KFS_06 --max-cutoffs-per-event 1`; reducing `--n-samples` changes Monte Carlo precision. Saved forests always run on CPU, even if a different tensor inference device is chosen.

**Plots are generated by default.** For both future new burns and total burned occupancy, `plots/` contains PNG and SVG figures for metrics, architecture distributions, all-model coverage, all nine probability-distance fields, reliability diagrams, event-level Brier scores, and ROC/precision–recall curves. The nine distance fields use eight panels because total variation equals Wasserstein-1 on the binary outcome space with unit ground distance. Every checkpoint remains in the metric, architecture and coverage overviews and in the CSV reports. Detailed panels use a documented display subset for legibility; this does not change evaluation cohorts or scores. Infinite and undefined exact scores are marked separately. `plots/model_index.csv` maps short identifiers to checkpoints, and `plots/manifest.json` records display selection and artifact hashes. Curves require the relevant positive/negative reference classes.

`report.md` embeds the plots and describes the recorded simulation assumptions. `report.json`, `summary.csv`, `per_event.csv`, `per_cutoff.csv` and `calibration.csv` contain scores, original reference coverage, both probability quantities, the shared-model comparison cohort, native-coverage diagnostics and all nine distance fields. Pooled metrics are recomputed from pooled predictions and outcomes. Add `--save-predictions` to retain complete rectangular probability, uncertainty, validity and coordinate arrays. Add `--no-plots` only when figures are not needed. These comparisons evaluate the repeated-current environmental scenario; Monte Carlo intervals quantify simulation error at individual cells, not uncertainty in reported evaluation metrics.


## 5. Plot a matrix and inspect its geographic placement

`result.plot(index=0)` selects the first time step **within this prediction**, displays `q = result.probability`, applies its validity mask, and returns a Matplotlib figure and axis. Here `result` has shape `(1, 1, H, W)` because it was predicted from one selected snapshot; another index is outside that result. Selecting a loaded event time is a separate operation, demonstrated above with `select_time(event_data, index=...)`. It fixes the probability scale at 0–1 by default. Unavailable predictions remain masked. Styling options are described in the [option reference](documents/api.md#plotting-and-event-comparison).

The second plot below places actual WGS84 cell centers. It preserves the curved geographic mesh rather than approximating it as an affine latitude/longitude rectangle. No online basemap is needed.


In [ ]:
fig, ax = result.plot(index=0, cmap="magma", title="Next-hour new-burn probability q")
fig.savefig(OUTPUT_DIR / "probability.png", bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
spatial = sample["pixel_valid"][0].numpy()
ax.scatter(coordinates[..., 1][spatial], coordinates[..., 0][spatial],
           c="#d1d5db", s=2, rasterized=True, label="Grid cells")
known = valid[0, 0] & spatial
points = ax.scatter(coordinates[..., 1][known], coordinates[..., 0][known],
                    c=probability[0, 0][known], s=12, vmin=0, vmax=1,
                    cmap="magma", rasterized=True)
ax.set(title="Next-hour new-burn probability q at Earth coordinates",
       xlabel="Longitude (degrees, WGS84)", ylabel="Latitude (degrees, WGS84)")
ax.set_aspect(1 / np.cos(np.deg2rad(np.nanmean(coordinates[..., 0]))))
fig.colorbar(points, ax=ax, label="Next-hour new-burn probability q")
fig.savefig(OUTPUT_DIR / "probability_coordinates.png", bbox_inches="tight")
plt.show()
plt.close(fig)


## 6. Sequences and batches: coordinates stay static

The two-step sequence below contains current inputs at two UTC cutoffs. Each predicted slice forecasts its own following hour; the model is not being rolled forward recursively. Probabilities have shape `(T, 1, H, W)`, while one `(H, W, 2)` coordinate matrix serves every time step.

`collate_fire_sequences` adds the batch dimension and handles temporal and spatial padding for varying event durations and grids. Here a batch of size one demonstrates the contract `(B, T, 1, H, W)` with coordinates `(B, H, W, 2)`. Different batch items may have different spatial grids and coordinate systems.

The sequence below is selected from the previously loaded full timeline. `DataLoader(..., collate_fn=collate_fire_sequences)` supplies standard batch iteration. An explicit `load_data(..., sequence_length=24)` instead returns fixed windows for repeated iteration.


In [ ]:
SECOND_CUTOFF = (datetime.fromisoformat(CUTOFF.replace("Z", "+00:00"))
                 + timedelta(hours=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
sequence = select_time(event_data, CUTOFF, SECOND_CUTOFF)
sequence_result = model.predict(sequence)
sequence_probability, sequence_coordinates = sequence_result.probability, sequence_result.coordinates
assert sequence_probability.shape == (2, 1, H, W)
assert sequence_coordinates.shape == (H, W, 2)
torch.testing.assert_close(sequence_coordinates, coordinate_tensor, rtol=0, atol=0, equal_nan=True)
torch.testing.assert_close(sequence_probability[0], probability_tensor[0], rtol=0, atol=0, equal_nan=True)

batch_loader = DataLoader([sequence], batch_size=1, collate_fn=collate_fire_sequences)
batch = next(iter(batch_loader))
batch_result = model.predict(batch)
batch_probability, batch_coordinates = batch_result.probability, batch_result.coordinates
assert batch_probability.shape == (1, 2, 1, H, W)
assert batch_coordinates.shape == (1, H, W, 2)
torch.testing.assert_close(batch_coordinates[0], coordinate_tensor, rtol=0, atol=0, equal_nan=True)
display(pd.DataFrame([
    {"case": "one snapshot", "prediction": tuple(probability_tensor.shape), "coordinates": tuple(coordinate_tensor.shape)},
    {"case": "two-step sequence", "prediction": tuple(sequence_probability.shape), "coordinates": tuple(sequence_coordinates.shape)},
    {"case": "batch of one sequence", "prediction": tuple(batch_probability.shape), "coordinates": tuple(batch_coordinates.shape)},
]))

### 6.1 Compare events at a specific loaded time index

Set `COMPARISON_TIME_INDEX` below to choose an hour in the **full loaded `event_data` timeline**. Select it first with `select_time(event_data, index=...)`, predict that one snapshot, then call `comparison_result.compare(event_data)`. The full timeline supplies the previous burned area at the exact cutoff and the new-burn label at the exact next-hour endpoint automatically. The default comparison puts next-hour probability `q = comparison_result.probability`, the translucent brown previous burned area, and hollow cyan next-hour new-burn markers on the same axes. The markers preserve the underlying `q` or unavailable gray inside. The axes retain the entire loaded grid, with the physical `pixel_valid` region outlined separately from probability validity. Set `marker_facecolor` to a color for filled markers, or supply `region_valid` and `region_color` to customize the loaded-region boundary. Set `mode="side_by_side"` or `mode="both"` when you want additional separate panels. The selected prediction has `T=1`, so omit its local index or use `index=0`.

The earlier `sequence_result` contains only two snapshots at `CUTOFF` and `CUTOFF + 1 hour`; its local indices are `0` and `1`. Calling `sequence_result.compare(sequence, index=10)` cannot select loaded timeline position 10 because that position is not in the two-step prediction.

`newly_burned_area[t]` labels arrivals during `(t − 1 hour, t]`; a forecast at `t` describes `(t, t + 1 hour]`. Automatic alignment uses the exact endpoint at `t + 1 hour` for next-hour new burns and the exact cutoff `t` for the previous cumulative burned area. Each layer retains its own validity mask; a missing endpoint does not erase valid probabilities or known previous burns. An absent marker does not establish a negative outcome where the reference is unknown. Binary targets are retrospective reconstructions, not hourly observed flame geometry. The second example below supplies both a caller-aligned bare target for local prediction index zero and its corresponding previous burned area.


In [ ]:
print("Sequence time steps:", sequence["features"].shape[0])
print("Prediction shape:", tuple(sequence_result.probability.shape))
print("len(sequence) counts dictionary fields, not time steps:", len(sequence))


In [ ]:
# This is an index into the FULL LOADED timeline, not the two-step sequence.
COMPARISON_TIME_INDEX = 10
comparison_sample = select_time(event_data, index=COMPARISON_TIME_INDEX)
comparison_result = model.predict(comparison_sample)
comparison_cutoff = pd.to_datetime(int(comparison_sample["time_utc"][0]), unit="s", utc=True)
print("Loaded comparison time index:", COMPARISON_TIME_INDEX,
      "| cutoff:", comparison_cutoff.isoformat())
print("Selected prediction shape:", tuple(comparison_result.probability.shape))
fig, axes = comparison_result.compare(
    event_data, mode="overlay", figsize=(7, 6),
    titles=f"Previous burned area, next-hour new burns and q\nCutoff: {comparison_cutoff.isoformat()}",
)
assert len(axes) == 1
np.testing.assert_allclose(axes[0].get_xlim(), (-0.5, W - 0.5))
np.testing.assert_allclose(axes[0].get_ylim(), (H - 0.5, -0.5))
fig.savefig(OUTPUT_DIR / "reconstructed_events_vs_probability.png", bbox_inches="tight")
plt.show()
plt.close(fig)

# The earlier TWO-STEP sequence has local time indices 0 and 1.
assert sequence_result.probability.shape[0] == 2
assert bool(sequence["time_valid"].all())
assert int(sequence["time_utc"][1] - sequence["time_utc"][0]) == 3600
aligned_target = sequence["newly_burned_area"][1, 0]
aligned_target_valid = sequence["burned_area_valid"][1, 0]
fig, axes = sequence_result.compare(
    aligned_target, observed_valid=aligned_target_valid,
    burned_area=sequence["burned_area"][0, 0],
    burned_area_valid=sequence["burned_area_valid"][0, 0],
    index=0, mode="overlay",
)
assert len(axes) == 1
np.testing.assert_allclose(axes[0].get_xlim(), (-0.5, W - 0.5))
np.testing.assert_allclose(axes[0].get_ylim(), (H - 0.5, -0.5))
fig.savefig(OUTPUT_DIR / "explicit_target_vs_probability.png", bbox_inches="tight")
plt.close(fig)


### 6.2 Spatial patches and padding

`patch_size` requests rectangular spatial patches without resizing the raster. The affine transform is shifted to the patch origin. In an edge patch, `pixel_valid=False` marks cells beyond the real grid and those cells receive NaN coordinates. Temporal padding retains the static grid and is represented by `time_valid` and the input masks.

In [ ]:
selection = dict(events=[EVENT], channels=DATA_CHANNELS, sequence_length=1,
                 time_range=(CUTOFF, CUTOFF))
with load_data(DATA_ROOT, **selection, patch_size=(32, 32)) as patches:
    edge_sample = patches[-1]
edge_result = model.predict(edge_sample)
edge_probability, edge_coordinates = edge_result.probability, edge_result.coordinates
edge_spatial = edge_sample["pixel_valid"][0].numpy()
assert edge_coordinates.shape == (32, 32, 2)
assert np.isfinite(edge_coordinates.numpy()[edge_spatial]).all()
assert np.isnan(edge_coordinates.numpy()[~edge_spatial]).all()
print("Edge patch spatial window [row, column, real height, real width]:",
      edge_sample["spatial_window"].tolist())
print("Real cells:", int(edge_spatial.sum()), "| spatially padded cells:", int((~edge_spatial).sum()))

## 7. Save and reload resident model parameters

`save` writes resident parameters with the saved normalization and model metadata. It does not fit or reload preprocessing. Existing destinations are protected unless `overwrite=True` is explicit. This notebook uses a fresh output directory on each execution.

In [ ]:
SAVED_MODEL = model.save(OUTPUT_DIR / "model.pt")
reloaded = load_model(SAVED_MODEL, device=DEVICE)
reloaded_result = reloaded.predict(sample)
reloaded_probability, reloaded_coordinates = reloaded_result.probability, reloaded_result.coordinates
torch.testing.assert_close(reloaded_probability, probability_tensor, rtol=0, atol=0, equal_nan=True)
torch.testing.assert_close(reloaded_coordinates, coordinate_tensor, rtol=0, atol=0, equal_nan=True)
assert reloaded.contract == model.contract
print("Saved checkpoint:", SAVED_MODEL)
print("Save/reload preserves probability, coordinates, and the model contract.")

## 8. Connect the wildfire snapshot to radiant heat

`WildfireModel.heat_model(...)` reuses the loaded wildfire weights and saved preprocessing. Its `predict_sample(sample)` method consumes **one spatial snapshot** and returns:

```python
heat_matrix, heat_coordinates = heat_model.predict_sample(sample)
```

Both values are NumPy arrays: `heat_matrix` has shape `(H, W)` and contains the **mean incident radiant heat flux over the next hour**, in W/m²; `heat_coordinates` has shape `(H, W, 2)` and contains receiver cell-center **`(latitude, longitude)`**, in WGS84 degrees. The coordinates have no time dimension. Use one heat call per snapshot when processing a sequence. This walkthrough continues to use the original **analytic one-hour** `result.probability`. Multi-hour first-new-burn probabilities and total burned occupancy do not specify ignition timing or active flame duration and must not be supplied directly to this one-hour heat bridge.

This example derives a **cell-specific representative flaming duration** from the aligned LANDFIRE 2016 FBFM40 raster. `FlameResidenceTime.from_fbfm40_raster(...)` combines the reference fuel classes with Scott–Burgan characteristic particle surface-area-to-volume ratios and Anderson's residence relation. It verifies the raster shape, affine grid and EPSG against the sample. Missing, nonburnable or unsupported codes require an explicit fallback; here we supply **2000 ft⁻¹**, converted to m⁻¹, and retain its per-cell mask and count. The reference classes and the fallback are scenarios, not measured event-specific extinguishment times.

For a known already-burned cell $i$, let $A_i$ be its supplied arrival time, $t_0$ the snapshot time, and $\tau_i>0$ the modeled duration, all in seconds from one event origin. Its current activity and remaining duration are

$$
p_i^0=\mathbf{1}\{A_i\le t_0<A_i+\tau_i\},\qquad
R_i=p_i^0\bigl(\tau_i-(t_0-A_i)\bigr).
$$

A location becomes inactive exactly at $A_i+\tau_i$. The heat sample API uses `arrival_hours`, `elapsed_hours` and their masks by default when available; `initial_burning_from_arrival=False` excludes that timing. Only known already-burned support uses these arrival values. Unknown age remains unresolved. New-burn inference continues to use the saved predictor, current state and environment.

Conditional on first new burning during the forecast interval of length $D=3600$ seconds, the heat conversion assumes a uniform arrival time and constant emission over $\tau_i$. For new-burn probability $q_i$ and $r_i=\min(\tau_i/D,1)$, the new-fire contributions are

$$
\overline p_i^{\mathrm{new}}=q_i(r_i-r_i^2/2),\qquad
p_i^{\mathrm{new}}(D)=q_i r_i.
$$

Existing fire adds $p_i^0\min(R_i,D)/D$ to mean activity and $p_i^0\mathbf{1}\{R_i>D\}$ to endpoint activity. These sources occupy known already-burned cells, disjoint from first new burning.

The data's arrivals are **retrospective point estimates with kriging uncertainty**, not exact flame-onset observations; that uncertainty is not integrated here. The duration describes a passing surface front at a representative point, excluding residual flaming, smouldering and within-cell propagation. Dry fuel loading, the consumed fraction **0.10**, effective heat yield and radiation parameters below remain illustrative scenarios. The conditional heat-release density is fuel loading × consumed fraction × effective heat yield / the **same cell-specific duration** used for activity.

In [ ]:
from wildfire.heat import AffineGrid
from wildfire.heat import CombustionParameters
from wildfire.heat import ExpectedHeatFluxModel
from wildfire.heat import FlameResidenceTime
import rasterio

heat_grid = AffineGrid(tuple(sample["affine_transform"].tolist()))
fuel_path = DATA_ROOT / "regions" / EVENT / "fuel" / "landfire_2016_fbfm40_375m.tif"
residence = FlameResidenceTime.from_fbfm40_raster(
    fuel_path,
    grid=heat_grid,
    expected_shape=(H, W),
    epsg=int(sample["epsg"]),
    fallback_sav_m_inv=2000.0 / 0.3048,  # Explicit scenario for unsupported/missing codes.
)
with rasterio.open(fuel_path) as fuel_source:
    fuel_codes = fuel_source.read(1, masked=True).astype(float).filled(np.nan)
catalog_codes_used = [int(code) for code in residence.provenance["catalog_codes_used"]]
duration_fallback_mask = ~np.isin(fuel_codes, catalog_codes_used)
assert int(duration_fallback_mask.sum()) == residence.provenance["fallback_cell_count"]

combustion = CombustionParameters.from_flame_residence(
    residence,
    fuel_load_kg_m2=1.0,
    heat_of_combustion_j_kg=18_000_000.0,
    consumed_fraction=0.10,
)
heat_model = model.heat_model(
    grid=heat_grid,
    epsg=int(sample["epsg"]),
    combustion=combustion,
    residence_time=residence,
    radiative_fraction=0.3,
    emission_height_m=5.0,
    receiver_height_m=0.0,
    atmospheric_transmissivity=1.0,
)
# The raw/probability bridges below use the same explicitly declared event clock.
arrival_valid = sample["arrival_valid"].detach().cpu().numpy().reshape(H, W)
arrival_s = sample["arrival_hours"].detach().cpu().numpy().reshape(H, W).astype(float) * 3600.0
arrival_s[~arrival_valid] = np.nan
current_s = float(sample["elapsed_hours"].item()) * 3600.0
print("Residence-duration range:", float(residence.duration_s.min()),
      "to", float(residence.duration_s.max()), "seconds")
print("Reference-catalog cells:", residence.provenance["catalog_cell_count"],
      "| explicitly supplied fallback cells:", int(duration_fallback_mask.sum()))
print("Conditional heat-release density range:",
      float(combustion.heat_release_rate_density_w_m2.min()), "to",
      float(combustion.heat_release_rate_density_w_m2.max()), "W/m²")
print("Forecast interval:", heat_model.interval_s, "seconds | source EPSG:", heat_model.epsg)


In [ ]:
heat_matrix, heat_coordinates = heat_model.predict_sample(
    sample, initial_burning_from_arrival=True,
)
heat_details = heat_model.predict_sample_details(
    sample, initial_burning_from_arrival=True,
)

assert heat_matrix.shape == probability.shape[-2:]
assert heat_coordinates.shape == (*heat_matrix.shape, 2)
np.testing.assert_allclose(heat_coordinates, coordinates, rtol=0, atol=0, equal_nan=True)
np.testing.assert_allclose(
    heat_matrix, heat_details.expected_heat_flux_w_m2,
    rtol=1e-12, atol=1e-8, equal_nan=True,
)
print("Heat matrix:", heat_matrix.shape, "W/m²")
print("Static coordinate matrix:", heat_coordinates.shape, "latitude/longitude degrees")
print("Complete supplied-domain mean expectation:", heat_details.complete)
print("Complete supplied-domain endpoint expectation:", heat_details.endpoint_complete)
print("Finite full-expectation receivers:", int(np.isfinite(heat_matrix).sum()))
print("Unresolved emitting source cells:",
      heat_details.metadata["unresolved_emitting_source_cells"])
print("Known modeled active cells at cutoff:", heat_details.metadata["known_initial_active_cells"])
print("Unresolved existing-fire ages:", heat_details.metadata["unresolved_initial_burning_cells"])
print("Radiation scope:", heat_details.metadata["scope"])


### Interpret missing heat and inspect activity diagnostics

In this FishCreek example, some possible emitting sources have unresolved new-burn probabilities. Therefore `heat_details.complete` is `False`, and the **full expected flux matrix is all `NaN`**. This includes receivers whose own wildfire probability is available: radiation from an unresolved source elsewhere in the supplied domain can reach those receivers. Missing existing-fire age can also make mean activity unresolved; its bounds retain that uncertainty.

`predict_sample_details(...)` returns the richer `IntegratedHeatPrediction`. The known-source contribution remains finite and equals the lower bound under the fixed nonnegative source assumptions. The upper bound permits unknown new-burn probabilities up to one and, where relevant, unknown age on known already-burned support. These are uncertainty endpoints, not missing observations replaced by measurements.

| Diagnostic | Definition | Unit |
| --- | --- | --- |
| `initial_active_burning_probability`, `initial_burning_valid` | Nominal current activity from arrival age and its validity | Probability / Boolean |
| `initial_remaining_burning_duration_s` | Remaining duration at the cutoff; unknown values stay `NaN` | Seconds |
| `mean_active_burning_probability`, `active_burning_valid` | Mean activity including continuing old fire and first new burning | Probability / Boolean |
| `endpoint_active_burning_probability`, `endpoint_active_burning_valid` | Activity at the end of the next hour and its validity | Probability / Boolean |
| `known_source_heat_flux_w_m2` | Contribution from known modeled sources | W/m² |
| `heat_flux_lower_bound_w_m2`, `heat_flux_upper_bound_w_m2` | Mean flux bounds under the fixed supplied assumptions | W/m² |
| `expected_endpoint_heat_flux_w_m2`, `endpoint_complete` | Full instantaneous endpoint expectation and its availability | W/m² / Boolean |
| `expected_incident_energy_j_m2` | Full mean expected flux × 3600 seconds | J/m² |
| `incident_energy_lower_bound_j_m2`, `incident_energy_upper_bound_j_m2` | Corresponding bounds on interval incident energy | J/m² |

The radiation model uses a finite supplied source domain and upward-facing point receivers at cell centers. These bounds exclude outside-domain fires, existing fire hidden by unknown cumulative-state coverage, convection, physical-parameter uncertainty, and unmodeled heat mechanisms. Incident radiant flux differs from absorbed heat or a cell-area average. A nominal inactive cell under the short residence scenario does not establish that residual real flames have extinguished.

In [ ]:
known_flux = heat_details.known_source_heat_flux_w_m2
lower_flux = heat_details.heat_flux_lower_bound_w_m2
upper_flux = heat_details.heat_flux_upper_bound_w_m2
assert np.isfinite(known_flux).all()
assert np.isfinite(lower_flux).all() and np.isfinite(upper_flux).all()
np.testing.assert_allclose(known_flux, lower_flux, rtol=0, atol=0)
assert np.all(lower_flux <= upper_flux + 1e-8)

flux_summary = pd.DataFrame([
    {"quantity": label, "unit": "W/m²", "finite_cells": int(np.isfinite(values).sum()),
     "minimum": float(np.min(values)) if np.isfinite(values).all() else np.nan,
     "maximum": float(np.max(values)) if np.isfinite(values).all() else np.nan}
    for label, values in [
        ("Full expected flux", heat_matrix),
        ("Known-source contribution = lower bound", lower_flux),
        ("Upper bound", upper_flux),
    ]
])
display(flux_summary)

np.testing.assert_allclose(
    heat_details.expected_incident_energy_j_m2,
    heat_matrix * heat_model.interval_s,
    rtol=1e-12, atol=1e-8, equal_nan=True,
)
np.testing.assert_allclose(
    heat_details.incident_energy_lower_bound_j_m2,
    lower_flux * heat_model.interval_s, rtol=1e-12, atol=1e-8,
)
np.testing.assert_allclose(
    heat_details.incident_energy_upper_bound_j_m2,
    upper_flux * heat_model.interval_s, rtol=1e-12, atol=1e-8,
)
print("Verified: interval incident energy = mean incident flux × interval seconds.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
for ax, values, title in zip(
    axes, (lower_flux, upper_flux),
    ("Known-source contribution / lower bound", "Upper bound under unresolved activity"),
):
    image = ax.imshow(values / 1000.0, origin="upper", cmap="inferno")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Raster column")
    ax.set_ylabel("Raster row")
    fig.colorbar(image, ax=ax, label="Mean incident radiant flux (kW/m²)", shrink=0.8)
fig.suptitle("FishCreek: reference-fuel duration and illustrative heat inputs; separate color scales")
heat_plot_path = OUTPUT_DIR / "heat_bounds.png"
fig.savefig(heat_plot_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

np.savez_compressed(
    OUTPUT_DIR / "heat_outputs.npz",
    complete=heat_details.complete,
    endpoint_complete=heat_details.endpoint_complete,
    mean_flux_w_m2=heat_matrix,
    endpoint_flux_w_m2=heat_details.expected_endpoint_heat_flux_w_m2,
    coordinates_latitude_longitude_deg=heat_coordinates,
    known_source_flux_w_m2=known_flux,
    lower_flux_w_m2=lower_flux,
    upper_flux_w_m2=upper_flux,
    initial_active_burning_probability=heat_details.initial_active_burning_probability,
    initial_burning_valid=heat_details.initial_burning_valid,
    initial_remaining_burning_duration_s=heat_details.initial_remaining_burning_duration_s,
    mean_active_burning_probability=heat_details.mean_active_burning_probability,
    active_burning_valid=heat_details.active_burning_valid,
    endpoint_active_burning_probability=heat_details.endpoint_active_burning_probability,
    endpoint_active_burning_valid=heat_details.endpoint_active_burning_valid,
    burning_duration_s=np.asarray(residence.duration_s),
    duration_fallback_mask=duration_fallback_mask,
    arrival_time_s=arrival_s,
    arrival_valid=arrival_valid,
    current_time_s=current_s,
    incident_energy_j_m2=heat_details.expected_incident_energy_j_m2,
    lower_incident_energy_j_m2=heat_details.incident_energy_lower_bound_j_m2,
    upper_incident_energy_j_m2=heat_details.incident_energy_upper_bound_j_m2,
)
print("Saved heat, current/endpoint activity, validity, duration and timing arrays to:", OUTPUT_DIR)


### Raw inputs and an already computed probability matrix

The heat API also accepts raw burned-area/environment arrays, or a previously computed new-burn probability matrix. The configured grid and projected EPSG supply Earth coordinates. Raw heat processing uses one spatial snapshot; any leading dimensions must be singleton.

The probability bridge below receives the original analytic **next-hour new-burn probabilities** in `[0, 1]`, their validity mask, and a known already-burned mask. It does not receive `cumulative_result.probability` or `burned_probability`. Rates in h⁻¹ must first be converted to probability by the model. Known already-burned cells have first-new-burn probability zero even where neighborhood context is unavailable.

Both routes below receive `arrival_time_s`, `current_time_s` and `arrival_valid` in the same event-origin clock as the sample route. They use arrivals only on known already-burned support, derive remaining duration from the configured combustion duration, and never convert retrospective future arrivals into forecast new burning. This explicit timing makes the sample, raw-input and probability-bridge results agree.

Separately supplied `initial_active_burning_probability` and `initial_remaining_burning_duration_s` remain available for independent current-fire information; these inputs replace arrival-derived timing and must refer to known already-burned cells. An `unknown_new_burn_probability` provides a separately justified estimate for unsupported forecast cells and is recorded separately in diagnostics. This demonstration leaves those cells unresolved.

In [ ]:
timing_options = {
    "arrival_time_s": arrival_s,
    "current_time_s": current_s,
    "arrival_valid": arrival_valid,
}
raw_heat, raw_heat_coordinates = heat_model.predict(
    sample["burned_area"], sample["features"],
    channel_names=sample["channel_names"],
    burned_area_valid=sample.get("burned_area_valid"),
    feature_valid=sample.get("feature_valid"),
    **timing_options,
)

# The full example has one time step and one prediction channel.
q_matrix = probability.reshape(heat_matrix.shape)
q_valid_matrix = valid.reshape(heat_matrix.shape)
burned_matrix = sample["burned_area"].detach().cpu().numpy().reshape(heat_matrix.shape)
burned_known = np.isfinite(burned_matrix)
if sample.get("burned_area_valid") is not None:
    burned_known &= sample["burned_area_valid"].detach().cpu().numpy().reshape(heat_matrix.shape)
already_burned = burned_known & (burned_matrix == 1)
bridge_options = {
    "probability_valid": q_valid_matrix,
    "already_burned_mask": already_burned,
    **timing_options,
}
bridged_heat, bridged_coordinates = heat_model.predict_from_new_burn_probability(
    q_matrix, **bridge_options,
)
bridged_details = heat_model.predict_from_new_burn_probability_details(
    q_matrix, **bridge_options,
)

for values in (raw_heat, bridged_heat):
    np.testing.assert_allclose(values, heat_matrix, rtol=0, atol=0, equal_nan=True)
for locations in (raw_heat_coordinates, bridged_coordinates):
    np.testing.assert_allclose(locations, heat_coordinates, rtol=0, atol=0, equal_nan=True)
for field in (
    "heat_flux_lower_bound_w_m2", "heat_flux_upper_bound_w_m2",
    "initial_active_burning_probability", "initial_remaining_burning_duration_s",
    "mean_active_burning_probability", "endpoint_active_burning_probability",
):
    np.testing.assert_allclose(
        getattr(bridged_details, field), getattr(heat_details, field),
        rtol=1e-12, atol=1e-8, equal_nan=True,
    )
for field in ("initial_burning_valid", "active_burning_valid", "endpoint_active_burning_valid"):
    np.testing.assert_array_equal(getattr(bridged_details, field), getattr(heat_details, field))
print("Verified: sample, raw-input and probability-bridge heat/activity routes agree.")


## 9. Standalone radiation without a wildfire checkpoint

`ExpectedHeatFluxModel.predict(active_burning_probability)` takes a complete, finite `(H, W)` matrix of **probabilities of active burning at the same instant**, all in `[0, 1]`. Its returned heat is instantaneous under those inputs; the composed API above instead returns an interval mean after converting next-hour new-burn probabilities into mean activity.

This independent example specifies a small **synthetic** grid in Korea's projected metric CRS `EPSG:5179`, with a 10 m cell size and explicit first-pixel-corner origin `(1,000,000 m, 2,000,000 m)`. Its grid and probabilities are illustrative, not observations. The returned receiver coordinates still use WGS84 `(latitude, longitude)` in degrees.

In [ ]:
active_probability = np.array([[0.0, 0.4, 0.0], [0.1, 0.8, 0.2]], dtype=np.float64)
synthetic_grid = AffineGrid.from_cell_size(
    cell_size_m=10.0,
    origin_xy_m=(1_000_000.0, 2_000_000.0),
    row_direction=-1,
)
# This small synthetic example has its own scalar scenario, independent of the real raster.
synthetic_combustion = CombustionParameters(
    fuel_load_kg_m2=1.0,
    heat_of_combustion_j_kg=18_000_000.0,
    burning_duration_s=120.0,
    consumed_fraction=1.0,
)
radiation = ExpectedHeatFluxModel(
    grid=synthetic_grid,
    epsg=5179,
    heat_release_rate_density_w_m2=synthetic_combustion.heat_release_rate_density_w_m2,
    radiative_fraction=0.3,
    emission_height_m=5.0,
    receiver_height_m=0.0,
    atmospheric_transmissivity=1.0,
)
instantaneous_flux, instantaneous_coordinates = radiation.predict(active_probability)
instantaneous_details = radiation.predict_details(active_probability)
assert instantaneous_flux.shape == active_probability.shape
assert instantaneous_coordinates.shape == (*active_probability.shape, 2)
np.testing.assert_allclose(
    instantaneous_flux, instantaneous_details.incident_heat_flux_w_m2, rtol=1e-12, atol=1e-8,
)
print("Instantaneous incident flux (W/m²):")
display(pd.DataFrame(instantaneous_flux))
print("Synthetic cell (0, 0) latitude/longitude:", instantaneous_coordinates[0, 0])
print("Expected total radiant source power:",
      instantaneous_details.expected_total_radiant_power_w, "W")

# This dose calculation adds a separate constant-flux-for-30-seconds assumption.
constant_flux_dose = instantaneous_details.incident_energy_j_m2(duration_s=30.0)
np.testing.assert_allclose(constant_flux_dose, instantaneous_flux * 30.0)

### Arbitrary receiver points

`predict_at_points` takes receiver **`(x, y)` in metres in the configured projected CRS**, not latitude/longitude. It returns `(flux_values, receiver_coordinates)` with shapes `(N,)` and `(N, 2)`. The latter uses WGS84 `(latitude, longitude)` in degrees. The following receivers are two synthetic cell centers, so their results should agree with the corresponding raster entries.

For a purely local metric grid without a known Earth reference, omit `epsg` and call `predict_details` or `predict_at_points_details`; the tuple methods require georeferencing.

In [ ]:
receiver_rows = np.array([0, 1])
receiver_columns = np.array([0, 1])
receiver_xy_m = synthetic_grid.centers(active_probability.shape)[receiver_rows, receiver_columns]
point_flux, point_coordinates = radiation.predict_at_points(active_probability, receiver_xy_m)
assert point_flux.shape == (2,) and point_coordinates.shape == (2, 2)
np.testing.assert_allclose(
    point_flux, instantaneous_flux[receiver_rows, receiver_columns],
    rtol=1e-10, atol=1e-8,
)
np.testing.assert_allclose(
    point_coordinates, instantaneous_coordinates[receiver_rows, receiver_columns],
    rtol=0, atol=1e-10,
)
display(pd.DataFrame({
    "row": receiver_rows,
    "column": receiver_columns,
    "x_projected_m": receiver_xy_m[:, 0],
    "y_projected_m": receiver_xy_m[:, 1],
    "latitude_deg": point_coordinates[:, 0],
    "longitude_deg": point_coordinates[:, 1],
    "incident_flux_w_m2": point_flux,
}))
print("Verified: point receivers agree with the corresponding raster cell centers.")

## 10. Inspect the duration-aware endpoint flame positions

`heat_details.endpoint_flame_map()` uses the **same residence scenario and continuing-fire timing** as the heat calculation. It estimates active burning **at the end of the next hour**, including any old fire whose remaining duration extends beyond that instant. This differs from the probability of new burning at any time during the hour. Current activity, remaining duration, mean activity and their masks remain in the exported arrays.

The durations come from the aligned reference-fuel raster and the explicitly recorded fallback. The arrivals are retrospective nominal estimates. Neither input establishes measured flame extinction or actual flame geometry. With seconds-long front residence, the modeled current or endpoint selection may be empty; that does not establish absence of real residual flames.

The probability threshold is a selection rule for displaying modeled cell centers; it is not a calibrated detection threshold. The endpoint returns projected positions separately. We look up WGS84 coordinates in the same static matrix.

In [ ]:
endpoint_flames = heat_details.endpoint_flame_map()
np.testing.assert_allclose(
    endpoint_flames.probability, heat_details.endpoint_active_burning_probability,
    rtol=0, atol=0, equal_nan=True,
)
MIN_ACTIVE_PROBABILITY = 0.001
positions = endpoint_flames.positions(min_probability=MIN_ACTIVE_PROBABILITY)
cells_rc = positions.cells_rc
selected_coordinates = coordinates[cells_rc[:, 0], cells_rc[:, 1]]
position_table = pd.DataFrame({
    "row": cells_rc[:, 0], "column": cells_rc[:, 1],
    "endpoint_active_probability": positions.probabilities,
    "latitude_deg": selected_coordinates[:, 0], "longitude_deg": selected_coordinates[:, 1],
    "duration_used_explicit_fallback": duration_fallback_mask[cells_rc[:, 0], cells_rc[:, 1]],
})
print("Selected endpoint cells:", len(position_table), "| threshold:", MIN_ACTIVE_PROBABILITY)
print("All positive nominal endpoint probabilities:",
      int(np.count_nonzero(endpoint_flames.valid & (endpoint_flames.probability > 0))))
display(position_table.head(10))


## 11. Export arrays and provenance, then verify the saved files

Export both predictions and coordinates together, retaining validity masks and source georeferencing. The source projected coordinates are `(x, y)` in the event's metric CRS; returned coordinates are `(latitude, longitude)` in WGS84 degrees. They are stored under different names.

The heat export includes the complete expectation, known-source contribution, bounds, incident energy, current and endpoint activity, remaining duration, the reference duration map, fallback mask and arrival clock. Unavailable values retain `NaN` with explicit validity masks. The manifest records the checkpoint checksum, input schema, timing, source CRS, physical scenarios, FBFM40 source checksum and residence provenance, library versions and checksums of generated artifacts. It does not certify hashes of every original source raster.

The cumulative NPZ separately includes first-new-burn probability, total burned probability, standard errors, Wilson bounds, validity, unavailable aggregate rates, horizon values and scalar Unicode `metadata_json`. Its readback requires no pickle support. The fixed-mode NPZ preserves analytic rates at fractional horizons. Strict cumulative evaluation JSON, a per-horizon metric/coverage/classification CSV and a separate calibration-bin CSV retain the explicitly selected durations and four quantity/coverage protocols.


In [ ]:
PROBABILITY_FILE = OUTPUT_DIR / "probability.npz"
HEAT_FILE = OUTPUT_DIR / "heat.npz"
probability_arrays = {
    "probability": probability, "valid": valid, diagnostic_name: diagnostic,
    "rate_valid": arrays["rate_valid"],
    "coordinates_latitude_longitude_deg": coordinates,
    "coordinate_epsg": np.asarray(4326, dtype=np.int64),
    "coordinate_order": np.asarray(["latitude", "longitude"]),
    "source_coordinates_xy_m": sample["coordinates"].numpy(),
    "source_epsg": sample["epsg"].numpy(),
    "affine_transform": sample["affine_transform"].numpy(),
    "time_utc": sample["time_utc"].numpy(), "time_valid": sample["time_valid"].numpy(),
    "pixel_valid": sample["pixel_valid"].numpy(),
}
with PROBABILITY_FILE.open("xb") as stream:
    np.savez_compressed(stream, **probability_arrays)
heat_arrays = {name: getattr(heat_details, name)
               for name in heat_details.__dataclass_fields__
               if isinstance(getattr(heat_details, name), np.ndarray)}
heat_arrays.update(coordinates_latitude_longitude_deg=heat_coordinates,
                   coordinate_epsg=np.asarray(4326, dtype=np.int64),
                   coordinate_order=np.asarray(["latitude", "longitude"]),
                   source_epsg=sample["epsg"].numpy(),
                   affine_transform=sample["affine_transform"].numpy(),
                   complete=np.asarray(heat_details.complete),
                   endpoint_complete=np.asarray(heat_details.endpoint_complete),
                   burning_duration_s=np.asarray(residence.duration_s),
                   duration_fallback_mask=duration_fallback_mask,
                   arrival_time_s=arrival_s, arrival_valid=arrival_valid,
                   current_time_s=np.asarray(current_s))
with HEAT_FILE.open("xb") as stream:
    np.savez_compressed(stream, **heat_arrays)

with np.load(PROBABILITY_FILE, allow_pickle=False) as saved:
    np.testing.assert_array_equal(saved["probability"], probability)
    np.testing.assert_array_equal(saved["coordinates_latitude_longitude_deg"], coordinates)
    assert saved["coordinates_latitude_longitude_deg"].shape == (H, W, 2)
with np.load(HEAT_FILE, allow_pickle=False) as saved:
    np.testing.assert_array_equal(saved['expected_heat_flux_w_m2'], heat_matrix)
    np.testing.assert_array_equal(saved["coordinates_latitude_longitude_deg"], coordinates)
    for field in ("initial_active_burning_probability", "initial_remaining_burning_duration_s",
                  "initial_burning_valid", "endpoint_active_burning_probability", "endpoint_active_burning_valid"):
        np.testing.assert_array_equal(saved[field], getattr(heat_details, field))
    np.testing.assert_array_equal(saved["burning_duration_s"], residence.duration_s)
    np.testing.assert_array_equal(saved["duration_fallback_mask"], duration_fallback_mask)
position_table.to_csv(OUTPUT_DIR / "illustrative_endpoint_flame_positions.csv", index=False)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "event": EVENT, "cutoff_utc": CUTOFF, "source_epsg": int(sample["epsg"]),
    "checkpoint": str(CHECKPOINT.relative_to(ROOT)), "checkpoint_sha256": model.checkpoint_sha256,
    "channel_names": list(model.channel_names), "model_contract": model.contract,
    "data_loading": {"sequence_length": None, "patch_size": None, "source_channels": list(DATA_CHANNELS),
                     "full_rectangle_diagnostics": grid_summary,
                     "full_timeline_feature_shape": list(event_data["features"].shape),
                     "full_timeline_start_utc": timeline_location["start_utc"],
                     "full_timeline_end_utc": timeline_location["end_utc"],
                     "valid_hours": timeline_location["valid_hours"]},
    "selected_comparison_time_index": COMPARISON_TIME_INDEX,
    "selected_comparison_cutoff_utc": comparison_cutoff.isoformat(),
    "cumulative_horizons_hours": list(cumulative_result.hours),
    "cumulative_comparison_hours": CUMULATIVE_COMPARISON_HOURS,
    "cumulative_assumption": "binary absorbing hourly trajectories; newly ignited sources spread next hour",
    "cumulative_method": cumulative_result.metadata["method"],
    "cumulative_metadata": cumulative_result.metadata,
    "cumulative_environment_policy": cumulative_result.metadata["environment_policy"],
    "cumulative_boundary_policy": {"boundary": BOUNDARY,
                                   "outside_grid_state": "unburned" if BOUNDARY == "zero" else "unknown",
                                   "output_region": "original rectangle",
                                   "physical_time_padding": "excluded",
                                   "missing_internal_environment": "unavailable"},
    "cumulative_controls": {"n_samples": N_SAMPLES, "seed": SEED,
                            "trajectory_batch_size": TRAJECTORY_BATCH_SIZE,
                            "missing_burned_area": MISSING_BURNED_AREA, "boundary": BOUNDARY},
    "cumulative_summary": cumulative_summary,
    "cumulative_fields": list(cumulative_arrays),
    "cumulative_forecast_example_executed": RUN_FORECAST_EXAMPLE,
    "cumulative_forecast_example_metadata": forecast_demo_metadata,
    "forest_example_executed": RUN_FOREST_EXAMPLE,
    "forest_example_controls": forest_demo_controls,
    "forest_example_metadata": forest_demo_metadata,
    "fixed_cumulative_horizons_hours": list(fixed_result.hours),
    "fixed_cumulative_metadata": fixed_result.metadata,
    "cumulative_evaluation_hours": list(EVALUATION_HOURS),
    "cumulative_evaluation_threshold": EVALUATION_THRESHOLD,
    "cumulative_evaluation_protocols": {name: report["protocol"] for name, report in cumulative_evaluations.items()},
    "cumulative_evaluation_coverage": {name: [{"hours": row["hours"], **row["coverage"]}
                                            for row in report["per_horizon"]]
                                      for name, report in cumulative_evaluations.items()},
    "cumulative_evaluation_report_summaries": {name: [
        {"hours": row["hours"], "evaluated_cells": row["coverage"]["evaluated_cells"],
         "positive_sample_count": row["metrics"]["positive_sample_count"],
         "brier": row["metrics"]["brier"],
         "exact_negative_log_likelihood_mean": row["metrics"]["exact_negative_log_likelihood_mean"],
         "bernoulli_kl_divergence_mean": row["metrics"]["bernoulli_kl_divergence_mean"],
         "f1": row["classification"]["f1"], "mcc": row["classification"]["mcc"],
         "weighted_expected_calibration_error": row["calibration"]["weighted_expected_calibration_error"]}
        for row in report["per_horizon"]] for name, report in cumulative_evaluations.items()},
    "cumulative_array_exports": ["cumulative_probability.npz", "fixed_cumulative_probability.npz"],
    "cumulative_evaluation_exports": ["cumulative_evaluation.json", "cumulative_evaluation.csv",
                                     "cumulative_calibration_bins.csv"],
    "cumulative_reference_policy": "original masks; unavailable references excluded; no initial contact restriction",
    "heat_probability_source": "original analytic next-hour result.probability",
    "cumulative_probability_shape": list(cumulative_result.probability.shape),
    "comparison_layout": "overlay",
    "comparison_marker_facecolor": "none",
    "comparison_grid_shape": [H, W],
    "comparison_region": "sample pixel_valid boundary; full grid extent",
    "comparison_layers": ["next-hour new-burn probability", "burned area at cutoff",
                          "reconstructed new burns in the following hour"],
    "selected_plot_time_index": PLOT_TIME_INDEX,
    "selected_plot_cutoff_utc": plot_cutoff.isoformat(),
    "probability_shape": list(probability.shape), "coordinate_shape": list(coordinates.shape),
    "coordinate_order": ["latitude", "longitude"], "coordinate_epsg": 4326,
    "known_prediction_cells": int(valid.sum()), "unknown_prediction_cells": int((~valid).sum()),
    "heat_complete": bool(heat_details.complete), "heat_metadata": heat_details.metadata,
    "heat_parameters": {
        "scenario": "reference-fuel residence and retrospective arrival; illustrative heat/consumption inputs",
        "fuel_load_kg_m2": np.asarray(combustion.fuel_load_kg_m2).tolist(),
        "heat_of_combustion_j_kg": np.asarray(combustion.heat_of_combustion_j_kg).tolist(),
        "burning_duration_s": {
            "minimum": float(residence.duration_s.min()), "maximum": float(residence.duration_s.max()),
            "array_file": "heat.npz", "array_key": "burning_duration_s",
        },
        "residence_time_provenance": residence.provenance_dict(),
        "consumed_fraction": np.asarray(combustion.consumed_fraction).tolist(),
        "radiative_fraction": np.asarray(heat_model.heat_model.radiative_fraction).tolist(),
        "emission_height_m": heat_model.heat_model.emission_height_m,
        "receiver_height_m": heat_model.heat_model.receiver_height_m,
        "atmospheric_transmissivity": heat_model.heat_model.atmospheric_transmissivity,
        "forecast_interval_s": heat_model.interval_s,
        "conditional_first_new_burn_time": "uniform within the forecast hour",
        "present_active_sources": "known burned-cell activity from nominal arrival age and modeled duration",
        "known_initial_active_cells": heat_details.metadata["known_initial_active_cells"],
        "explicit_duration_fallback_cells": int(duration_fallback_mask.sum()),
    },
    "python": sys.version.split()[0],
    "versions": {p: version(p) for p in ["numpy", "torch", "pandas", "pyproj", "rasterio"]},
    "artifacts": {p.name: {"sha256": sha256(p), "bytes": p.stat().st_size}
                  for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()},
    "verified": ["independent complete-event timeline loading", "exact loaded-time selection", "loaded index selection and plotting",
                 "standard DataLoader batching", "raw/sample probability equivalence", "static coordinates across T",
                 "batch coordinate alignment", "exact-UTC next-hour event comparison",
                 "previous burned area and next-hour events in one probability plot",
                 "requested-horizon cumulative probability and exact-endpoint comparison",
                 "recursive cumulative fields, metadata and NPZ readback",
                 "exact initial-burn endpoints and nondecreasing shared cumulative estimates",
                 "same-input analytic one-hour versus Monte Carlo comparison when requested",
                 "fixed fractional-horizon analytic probability and rate equivalence",
                 "explicit evaluation hours and method/standalone API equivalence",
                 "four separate new-burn/total-burned per-horizon/shared protocols",
                 "strict evaluation JSON, CSV and calibration-bin readback",
                 "full original rectangle and environmental support outside missing arrival labels",
                 "spatial-padding coordinates",
                 "save/reload equivalence", "wildfire/heat coordinate alignment",
                 "heat tuple/details equivalence", "sample/raw/probability-bridge activity equivalence",
                 "endpoint flame-map consistency", "current/endpoint activity and duration export readback"],
    "limitations": ["retrospective reconstructed current burned state",
                    "unavailable initial burn states assumed unburned only in recursive inference; original reference masks preserved",
                    "repeated-current environmental scenario unless caller forecasts supplied",
                    "conditional independent ignitions and persistent Bayesian components define the simulated joint process",
                    "pointwise Monte Carlo intervals exclude model, weather and evaluation-metric uncertainty",
                    "total-burned metrics include deterministic initial burns",
                    "reference-fuel front residence is not measured event-specific extinguishment",
                    "explicit duration fallback on missing/nonburnable/unsupported fuel codes",
                    "retrospective point-arrival uncertainty not integrated",
                    "illustrative fuel consumption and heat parameters",
                    "incomplete full-domain heat expectation",
                    "not a forecast-skill or measured-heat validation"],
}
manifest = json.loads(json.dumps(manifest, allow_nan=False))  # Normalize tuples for exact JSON readback.
MANIFEST_FILE = OUTPUT_DIR / "manifest.json"
MANIFEST_FILE.write_text(json.dumps(manifest, indent=2, allow_nan=False) + "\n")
display(pd.DataFrame([{"file": p.name, "bytes": p.stat().st_size}
                      for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()]))
print("Verified output directory:", OUTPUT_DIR)
assert json.loads(MANIFEST_FILE.read_text(), parse_constant=reject_nonstandard_json) == manifest


## 12. Optional: prepare training data and fit a model

Data preparation and model fitting are separate operations. `prepare_training_data(...)` reads full event timelines by default, with `chunk_hours=None`, through the independent loader. It extracts exact next-hour targets, preserves validity and inverse sampling weights, and writes explicit train/validation/test partitions. Set `chunk_hours=24` for bounded raster windows covering the same timeline. `train_model(prepared_data, output_dir, config=...)` fits those partitions; an existing cache is reused without rereading event rasters.

Both switches below are false during the main tutorial. Set `RUN_PREPARATION=True` to build a new cache in this execution's output directory. Set `RUN_TRAINING=True` to fit either that new cache or the included existing cache. The three-epoch example demonstrates execution and does not reproduce the saved experiment or establish model performance.

The trainer fits normalization using training rows, selects its checkpoint on validation, and reports test metrics afterward. The default weighted split keeps training and validation cell-hour rows disjoint and test fires separate. Categorical balancing adjusts training loss only; original population weights remain in validation and test metrics. Keep output directories new and preserve the recorded memberships.


In [ ]:
from wildfire import TrainingConfig

RUN_PREPARATION = False
RUN_TRAINING = False
PREPARED = ROOT / 'runs/20261009_weighted_stratified_split/prepared_all'
if RUN_PREPARATION:
    PREPARED = prepare_training_data(
        OUTPUT_DIR / "prepared", data_root=DATA_ROOT, channels=DATA_CHANNELS,
        kernel_size=5, chunk_hours=None,  # Full event timelines for training preparation.
        split_policy="weighted-cell-hour", validation_fraction=0.2,
        seed=7, split_seed=7,
    )
    print("New prepared training cache:", PREPARED)
else:
    print("Preparation skipped: RUN_PREPARATION=False; existing cache selected.")

required_cache_files = ("metadata.json", "train.npz", "validation.npz", "test.npz")
if RUN_TRAINING:
    assert all((PREPARED / name).is_file() for name in required_cache_files)
    trained = train_model(
        prepared_data=PREPARED, output_dir=OUTPUT_DIR / "fitted", family="exponential",
        config=TrainingConfig(epochs=3, patience=2, batch_size=1024, device="cpu", threads=4,
                              categorical_balance="inverse-sqrt",
                              categorical_balance_min_frequency=0.001,
                              categorical_balance_max_factor=10.0),
    )
    trained.save(OUTPUT_DIR / "trained_model.pt")
    print("Returned metric sections:", list(trained.training_metrics))
else:
    print("Optional training skipped: RUN_TRAINING=False. No model was fitted by this execution.")


## 13. API reference and interpretation

| Task | Public interface |
| --- | --- |
| Load the default or an explicit checkpoint | `load_model(path=DEFAULT_CHECKPOINT, device="cpu")` |
| Load a saved CPU patch forest | `load_model("forest.joblib", device="cpu")`; one-hour `predict()` and recursive `predict_cumulative()` |
| List catalog fire-event identifiers | `available_events(...)` |
| List an event's eligible hourly UTC cutoffs | `available_cutoffs(event, ...)` |
| Open a windowed dataset independently | `load_data(..., sequence_length=24)` |
| Load the full original event rectangle and timeline | `load_data(..., sequence_length=None, patch_size=None)` then `data[index]` |
| Select already-loaded times without reading files | `select_time(sample, start, end=None)` or `select_time(sample, index=i)` |
| Predict probability, masks, rate and optional coordinates | `model.predict(sample)` or `model.predict(burned_area, environmental_data, ...)` |
| Recursively predict caller-entered whole-hour cumulative probabilities | `model.predict_cumulative(sample, hours=HORIZONS_HOURS, n_samples=1024, seed=0, trajectory_batch_size=32)` |
| Control missing initial burn states | `missing_burned_area="unburned"` or `"unknown"`; environment/padding stay unavailable |
| Control exterior source padding in recursive mode | `boundary="zero"` (default outside-unburned) or `"unknown"`; output rectangle unchanged; physical/time padding excluded |
| Supply hourly environmental forecasts | `environmental_forecast={"features": hourly_features, "channel_names": names, "feature_valid": mask}` |
| Preserve analytic fixed/fractional-horizon compatibility | `model.predict_cumulative(sample, hours=[0.5, 1, 3.5], method="fixed")` |
| Evaluate explicitly entered cumulative durations | `cumulative_result.evaluate(reference, hours=EVALUATION_HOURS, quantity="new_burn", coverage="shared")` |
| Use the standalone public evaluator | `evaluate_cumulative(cumulative_result, reference, hours=EVALUATION_HOURS, quantity="burned")` |
| Compare every saved tensor/forest model and create PNG/SVG report plots | `.venv/bin/python -m wildfire evaluate-cumulative --hours 6 --models-root ...` |
| Select or plot one requested horizon | `cumulative_result.at(hours=6)` / `.plot(hours=6, index=...)` |
| Compare cumulative interval new burns, previous area and q | `cumulative_result.compare(timeline, hours=6, index=...)` |
| Obtain independent CPU NumPy fields and scalar JSON metadata | `result.numpy()` / `cumulative_result.numpy()` |
| Plot the selected probability matrix | `result.plot(index=...)` |
| Show previous burned area, next-hour new burns and probabilities together | `result.compare(sample, index=...)` or `result.compare(target, burned_area=current_state, ...)` |
| Plot a generic matrix or aligned event/probability matrices | `plot_matrix(...)` / `plot_comparison(...)` |
| Retrieve an explicit probability/coordinate tuple | `model.predict_coordinates(...)` |
| Move resident weights and normalization | `model.to(device)` |
| Save resident parameters | `model.save(path, overwrite=False)` |
| Compose wildfire and heat models | `model.heat_model(...)` / `WildfireHeatModel.from_checkpoint(...)` |
| Predict mean radiant heat and coordinates | `heat_model.predict_sample(...)`, `predict(...)`, `predict_from_new_burn_probability(...)` |
| Retrieve heat bounds, dose and provenance | Corresponding heat `*_details(...)` methods |
| Compute instantaneous radiation from complete active probabilities | `ExpectedHeatFluxModel.predict(...)` / `predict_at_points(...)` |
| Estimate next-hour active-flame locations | `model.predict_active_flames_sample(...)` |
| Prepare training partitions from full event timelines | `prepare_training_data(output_dir, chunk_hours=None, ...)` |
| Fit from a prepared cache | `train_model(prepared_data, output_dir, config=TrainingConfig(...))` |

The [complete option reference](documents/api.md) lists every data, inference, plotting, training, save, flame and heat option. Earlier `predict_sample`, `predict_details`, `predict_sample_details` and `predict_proba` methods remain adapters for existing callers; `predict` returns the new object.

Recursive inference may explicitly initialize unavailable initial burn states as unburned inside real support; this is an assumption recorded in metadata, not an observation. Missing environmental inputs, padding and reference outcomes stay unavailable. The evaluator always retains original reference masks. First-new-burn `probability` and total `burned_probability` are distinct, and recursive rate diagnostics are unavailable. First new cumulative burning, instantaneous flame activity, interval-mean heat flux (W/m²), and interval incident energy (J/m²) are different quantities. Positions are representative grid centers under the saved geometry and supplied physical assumptions.

For alternate checkpoints, events and scientific source discussion, use the [root README](README.md), [data contracts](README.md#data-guide), [saved reports](README.md#completed-reports), and [radiation references](wildfire/heat_sources.json).
